# Ajuste fino de la U-Net v4 con las teselas marcadas de La Convención

Parte de la U-Net v4 (IoU 0.460 en la prueba de Landslide4Sense) y la sigue entrenando con
tasa baja, mezclando en cada lote parches de La Convención y parches de Landslide4Sense para
que no olvide lo que ya sabe. Mide el antes y el después en zonas de La Convención que el
modelo no ve al entrenar.

**Qué tiene que haber en tu Drive antes de empezar**

| Qué | De dónde sale |
|---|---|
| `entrenamiento_v4/mejor_unet_b1b4b13b14_b2809be440d6.pt` y `stats_norm_b2809be440d6.json` | Los dejó el cuaderno `Entrenar_Modelos_v4` |
| `datos_locales_ajuste.zip` | En tu PC: `python herramientas/empaquetar_para_colab.py` y lo subes a Drive |

**Reglas que el cuaderno respeta solo**

- Solo entrena con máscaras de nivel 1 (las que ustedes marcaron). Las propuestas de INGEMMET
  (nivel 2) entran únicamente si pones `USAR_DEBILES = True`, con peso bajo y nunca para medir.
- La partición es por zona: ninguna zona está a la vez en entrenamiento y validación.
- La normalización no cambia: el modelo ajustado usa la misma receta de entrada que el servidor.
- El modelo base también compite. Si ninguna época lo supera sin perder en Landslide4Sense,
  el cuaderno lo dice y no exporta nada.

**Cómo leer el resultado.** La validación local se usa para elegir la época y el umbral, así que
su IoU es optimista. Con menos de unas 10 cicatrices en validación el resultado es exploratorio:
sirve para decidir si seguir marcando, no para la tesis.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU T4**. Tarda unos
10 minutos de descarga y entre 10 y 30 minutos de ajuste.

In [ ]:
!pip -q install segmentation-models-pytorch onnx onnxruntime h5py huggingface_hub rasterio scipy

## Configuración

In [ ]:
CARPETA_BASE = '/content/drive/MyDrive/entrenamiento_v4'      # donde está el modelo base
FIRMA_BASE = 'b2809be440d6'                                   # U-Net v4, semilla 42
ZIP_LOCAL = '/content/drive/MyDrive/datos_locales_ajuste.zip' # lo que armó empaquetar_para_colab.py
SALIDA = '/content/drive/MyDrive/ajuste_fino_laconvencion'    # puntos de control y resultados
AUTOR = 'Mejía López, L. C.; Huarcaya Pumacayo, V. N.'

EPOCAS = 40
TASA = 3e-5                # diez veces menor que en el entrenamiento original
FRACCION_LOCAL = 0.5       # parte de cada lote que es de La Convención
CONGELAR_ENCODER = False   # True = solo se ajusta el decodificador (más seguro con muy pocos datos)
USAR_DEBILES = True        # True = entrena con los parches Nivel 2 clasificados por Tiers (Oro/Plata/HardNeg)
PESO_DEBIL = 0.3           # peso base de respaldo si no hay catalogo_parches_tiers.csv
USAR_GEOMORFOLOGIA = False # True = activa CURVATURA y TPI desde el DEM (Vía 3)
TOLERANCIA_L4S = 0.03      # IoU que se acepta perder en la validación de Landslide4Sense


In [ ]:
import os, glob, json, shutil, subprocess, sys, zipfile
import torch
assert torch.cuda.is_available(), 'activa la GPU: Entorno de ejecución > Cambiar tipo de entorno > T4'
print('GPU:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(SALIDA, exist_ok=True)
CK_BASE = f'{CARPETA_BASE}/mejor_unet_b1b4b13b14_{FIRMA_BASE}.pt'
STATS_BASE = f'{CARPETA_BASE}/stats_norm_{FIRMA_BASE}.json'
for ruta in (CK_BASE, STATS_BASE, ZIP_LOCAL):
    assert os.path.exists(ruta), f'no está {ruta}: revisa la tabla de arriba'
print('modelo base y datos locales encontrados')

## El código del proyecto

Copiado sin cambios de `nucleo` y `herramientas`. La escala de bandas es la misma del servidor.

In [ ]:
import os
ARCHIVOS = {}
ARCHIVOS['nucleo/__init__.py'] = ''
ARCHIVOS['nucleo/config.py'] = '"""\nTodo lo configurable del backend, en un solo sitio.\n\nCada valor se puede cambiar con una variable de entorno del mismo nombre, que es\ncomo se ajusta en Render (Environment) sin tocar el código.\n"""\nimport json\nimport os\nimport tempfile\n\n\ndef _env(nombre, defecto):\n    return os.environ.get(nombre, defecto)\n\n\nRAIZ = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))\n\n# --------------------------------------------------------------------- modelos\n# Cada modelo es un .onnx más una receta .json con el mismo nombre de base, que\n# dice qué canales recibe, cómo se normalizan, qué salida tiene y su umbral.\nCARPETA_MODELOS = _env(\'CARPETA_MODELOS\', os.path.join(RAIZ, \'modelos\'))\n# Si un .onnx llega como puntero de Git LFS, se baja de aquí: <URL>/<archivo>\nURL_PESOS_BASE = _env(\'URL_PESOS_BASE\', \'\')\n# Un ONNX con pesos de inicialización no detecta nada aunque corra. Se rechaza\n# salvo que esto valga 1, que solo sirve para probar el circuito.\nPERMITIR_SIN_ENTRENAR = _env(\'PERMITIR_SIN_ENTRENAR\', \'0\') == \'1\'\nHILOS_ONNX = int(_env(\'HILOS_ONNX\', \'1\'))\n# Optimización del grafo: \'extendida\' da los mismos resultados (diferencia < 1e-6)\n# que \'todas\' y usa unos 100 MB menos, porque no duplica los pesos en el formato\n# interno NCHWc. En 512 MB eso decide si el servicio cabe o no.\nNIVEL_OPT_ONNX = _env(\'NIVEL_OPT_ONNX\', \'extendida\')\n# Hilos para bajar las imágenes. dask usa por defecto uno por núcleo de la\n# máquina, y en Render la máquina física tiene muchos aunque el plan dé 0.1 CPU:\n# cada hilo abre sus propias conexiones y cachés. Con 4 la descarga es rápida\n# (la espera es de red, no de cálculo) y la memoria queda acotada.\nHILOS_DESCARGA = int(_env(\'HILOS_DESCARGA\', \'4\'))\n# En el plan gratuito de Render hay 512 MB: mejor un modelo cargado a la vez.\nUN_MODELO_A_LA_VEZ = _env(\'UN_MODELO_A_LA_VEZ\', \'1\') == \'1\'\n# La sesión se carga recién para inferir y se suelta al terminar cada trabajo, así\n# no ocupa memoria mientras se bajan las imágenes. Con más RAM se puede poner 0.\nSOLTAR_SESION = _env(\'SOLTAR_SESION\', \'1\') == \'1\'\n# Parches por llamada al modelo. 4 da la misma velocidad que 8 en CPU y ~20 MB menos.\nLOTE_INFERENCIA = int(_env(\'LOTE_INFERENCIA\', \'4\'))\n# Al revisar cada modelo, el servidor guarda aquí una copia con los pesos en un\n# archivo aparte (.onnx.data). ONNX Runtime los lee directo del disco: unos 95 MB\n# dejan de contar como memoria propia del proceso y el resultado es idéntico.\n# Vacío para desactivarlo.\nCARPETA_COPIAS = _env(\'CARPETA_COPIAS\', os.path.join(\n    tempfile.gettempdir(), \'modelos_pesos_aparte\'))\n\n# ----------------------------------------------------------------------- CORS\n# La URL del Space, p. ej. https://usuario-espacio.hf.space. \'*\' para probar.\nORIGENES = [o.strip() for o in _env(\'ORIGENES_PERMITIDOS\', \'*\').split(\',\') if o.strip()]\n\n# ---------------------------------------------------------- fuente de imágenes\nSTAC = \'https://planetarycomputer.microsoft.com/api/stac/v1\'\nCOLECCION_S2 = \'sentinel-2-l2a\'\n# ----------------------------------------------------- relieve (B13 y B14)\n# Landslide4Sense toma la pendiente (B13) y el DEM (B14) del «ALOS PALSAR DEM» de\n# 12.5 m que distribuye ASF con los productos RTC. Ese DEM no sale de PALSAR: es\n# SRTM de 30 m remuestreado a 12.5 m (NASA Earthdata, proyecto ALOS PALSAR RTC).\n# Por eso la fuente por defecto, que cubo pide a Planetary Computer, es NASADEM\n# (SRTM reprocesado, 30 m). Otras opciones:\n#   alos-aw3d30   ALOS World 3D-30m de JAXA (óptico PRISM, misión ALOS)\n#   cop-dem-glo-30 Copernicus GLO-30\n#   asf-alos-palsar el mosaico de los .dem.tif de ASF para la provincia, armado\n#                  una vez con herramientas/alos_palsar_asf.py (archivo o URL COG)\n# Ninguna es una serie temporal: cada una va con su propia ventana de fechas.\nFUENTES_DEM = {\n    \'nasadem\': {\'coleccion\': \'nasadem\', \'banda\': \'elevation\',\n                \'fechas\': (\'2000-01-01\', \'2000-12-31\'),\n                \'nombre\': \'NASADEM (SRTM, 30 m), la base del DEM «ALOS PALSAR» de ASF\'},\n    \'alos-aw3d30\': {\'coleccion\': \'alos-dem\', \'banda\': \'data\',\n                    \'fechas\': (\'2016-01-01\', \'2016-12-31\'),\n                    \'nombre\': \'ALOS World 3D-30m (JAXA, 30 m)\'},\n    \'cop-dem-glo-30\': {\'coleccion\': \'cop-dem-glo-30\', \'banda\': \'data\',\n                       \'fechas\': (\'2020-01-01\', \'2023-12-31\'),\n                       \'nombre\': \'Copernicus DEM GLO-30 (30 m)\'},\n    \'asf-alos-palsar\': {\'archivo\': True,\n                        \'nombre\': \'ALOS PALSAR RTC (ASF), mosaico de la provincia\'},\n}\nFUENTE_DEM = _env(\'FUENTE_DEM\', \'nasadem\')\nif FUENTE_DEM not in FUENTES_DEM:\n    raise ValueError(f\'FUENTE_DEM="{FUENTE_DEM}" no es una de {sorted(FUENTES_DEM)}\')\n# Para asf-alos-palsar: ruta local o URL (https://...) del GeoTIFF del mosaico.\nRUTA_DEM_ASF = _env(\'RUTA_DEM_ASF\', \'\')\n# Constantes de Landslide4Sense, medidas en sus parches (son las mismas en todas\n# las regiones): pendiente = grados / 20.292 y elevación = metros / 772.82.\nESCALA_TERRENO_L4S = {\'pendiente\': 20.292, \'elevacion\': 772.82}\nRESOLUCION_M = 10\n# Lado de cada cubo de cubo. 256 px son 2.56 km: pocas llamadas y poca memoria.\nLADO_TESELA_PX = int(_env(\'LADO_TESELA_PX\', \'256\'))\n# Escenas por tesela, las menos nubosas según el metadato. Acota memoria y tiempo.\nMAX_ESCENAS = int(_env(\'MAX_ESCENAS\', \'6\'))\n# Clases SCL que se descartan: sin dato, saturado, sombra de nube, nube media,\n# nube alta y cirro.\nSCL_MALAS = (0, 1, 3, 8, 9, 10)\n# Desde la baseline 04.00 (enero de 2022) los niveles digitales de L2A llevan\n# sumado 1000. Se resta por escena.\nDESPLAZAMIENTO_L2A = 1000.0\n\n# ---------------------------------------------- unidades de Landslide4Sense\n# Landslide4Sense trae cada banda dividida por su media de escena. Estas son las\n# medias de La Convención (reflectancia L2A en niveles digitales), medidas con\n# herramientas/calibrar_escala.py sobre la provincia en temporada seca.\n_RUTA_ESCALA = _env(\'RUTA_ESCALA\', os.path.join(RAIZ, \'escala_laconvencion.json\'))\n\n\ndef _cargar_escala():\n    try:\n        with open(_RUTA_ESCALA, encoding=\'utf-8\') as f:\n            return json.load(f)[\'media_dn\']\n    except Exception:\n        return None\n\n\nESCALA_L4S = _cargar_escala()\n\n# ------------------------------------------------------------------- límites\nMAX_LADO_KM = float(_env(\'MAX_LADO_KM\', \'12\'))\nAVISO_LADO_KM = float(_env(\'AVISO_LADO_KM\', \'6\'))\nSOLAPE = float(_env(\'SOLAPE\', \'0.5\'))\nCOBERTURA_MIN = float(_env(\'COBERTURA_MIN\', \'0.30\'))\nMAX_COLA = int(_env(\'MAX_COLA\', \'4\'))\nMINUTOS_RESULTADO = int(_env(\'MINUTOS_RESULTADO\', \'30\'))\n\n# -------------------------------------------------------------------- ámbito\n#   \'bloquear\': rechaza con 403 lo que caiga fuera. Es lo que va en la entrega.\n#   \'avisar\'  : deja correr y marca el resultado como fuera del ámbito.\n#   \'libre\'   : sin límite, solo para pruebas.\nAMBITO = _env(\'AMBITO\', \'bloquear\')\nNOMBRE_AMBITO = \'provincia de La Convención, Cusco\'\nRUTA_LIMITE = _env(\'RUTA_LIMITE\', os.path.join(RAIZ, \'limite_laconvencion.geojson\'))\nBBOX_ESTUDIO = (-13.4489, -73.9805, -11.1988, -71.9730)   # sur, oeste, norte, este\nMIN_DENTRO = float(_env(\'MIN_DENTRO\', \'0.5\'))\n\n# --------------------------------------------------------------------- pruebas\n# A 1 no sale a la red: fabrica un área sintética con una cicatriz.\nMODO_FALSO = _env(\'MODO_FALSO\', \'0\') == \'1\'\nCENTRO_POR_DEFECTO = (-12.86, -72.69)    # Quillabamba\n\n# Zonas marcadas más chicas que esto (en píxeles de 10 m) no se listan en el GeoJSON.\n# En la prueba de Landslide4Sense, 8 de cada 10 zonas de 1 a 3 px son falsas alarmas.\nZONA_MIN_PX = int(_env(\'ZONA_MIN_PX\', \'4\'))\n'
ARCHIVOS['nucleo/recetas.py'] = '"""\nRecetas de entrada de cada modelo.\n\nUn ONNX no dice qué espera en cada canal. Eso va en una receta JSON junto al\n.onnx, con el mismo nombre de base:\n\n    modelos/deeplabv3plus.onnx\n    modelos/deeplabv3plus.json\n\nLa receta fija los canales y su orden, la normalización con sus constantes, el\ntipo de salida y el umbral. La función `armar` construye el tensor a partir de\ncanales ya expresados en unidades de Landslide4Sense, y la usan por igual el\nbackend (con imágenes de cubo) y la verificación (con los h5 de L4S). Así lo que\nse mide contra Landslide4Sense es exactamente lo que corre en producción.\n"""\nimport json\nimport os\n\nimport numpy as np\n\nfrom . import config\n\n# Posición de cada canal en los h5 de Landslide4Sense (128 x 128 x 14).\nIDX_L4S = {\'B01\': 0, \'B02\': 1, \'B03\': 2, \'B04\': 3, \'B05\': 4, \'B06\': 5, \'B07\': 6,\n           \'B08\': 7, \'B09\': 8, \'B10\': 9, \'B11\': 10, \'B12\': 11,\n           \'pendiente\': 12, \'elevacion\': 13}\nINDICES = {\'NDVI\': (\'B08\', \'B04\'), \'NDWI\': (\'B03\', \'B08\'), \'GRVI\': (\'B03\', \'B04\')}\n# Nombres de Landslide4Sense para los canales de relieve: B13 = pendiente y\n# B14 = DEM. Una receta puede usar cualquiera de los dos nombres.\nALIAS = {\'B13\': \'pendiente\', \'B14\': \'elevacion\', \'B1\': \'B01\', \'B2\': \'B02\', \'B3\': \'B03\',\n         \'B4\': \'B04\', \'B5\': \'B05\', \'B6\': \'B06\', \'B7\': \'B07\', \'B8\': \'B08\', \'B9\': \'B09\'}\nTERRENO = (\'pendiente\', \'elevacion\')\nDERIVADOS_TERRENO = (\'CURVATURA\', \'TPI\')\nSALIDAS = (\'sigmoide\', \'softmax\')\n\n\nclass RecetaInvalida(ValueError):\n    pass\n\n\ndef normalizar_nombres(r):\n    """B13 -> pendiente, B14 -> elevacion, B1 -> B01... en canales y constantes."""\n    al = lambda c: ALIAS.get(c, c)\n    r[\'canales\'] = [al(c) for c in r[\'canales\']]\n    n = r.get(\'normalizacion\', {})\n    if \'canales\' in n:\n        n[\'canales\'] = [al(c) for c in n[\'canales\']]\n    if \'por_canal\' in n:\n        n[\'por_canal\'] = {al(c): v for c, v in n[\'por_canal\'].items()}\n    if \'escala_terreno\' in r:\n        r[\'escala_terreno\'] = {al(c): v for c, v in r[\'escala_terreno\'].items()}\n    if \'rango_entrenamiento\' in r:\n        r[\'rango_entrenamiento\'] = {al(c): v for c, v in r[\'rango_entrenamiento\'].items()}\n    return r\n\n\ndef leer(ruta_json):\n    with open(ruta_json, encoding=\'utf-8\') as f:\n        r = json.load(f)\n    normalizar_nombres(r)\n    validar(r, os.path.basename(ruta_json))\n    return r\n\n\ndef validar(r, origen=\'receta\'):\n    for campo in (\'nombre\', \'onnx\', \'canales\', \'lado_px\', \'normalizacion\', \'salida\',\n                  \'umbral\'):\n        if campo not in r:\n            raise RecetaInvalida(f\'{origen}: falta "{campo}"\')\n    for c in r[\'canales\']:\n        if c not in IDX_L4S and c not in INDICES and c not in DERIVADOS_TERRENO:\n            raise RecetaInvalida(f\'{origen}: canal desconocido "{c}"\')\n    if r[\'salida\'] not in SALIDAS:\n        raise RecetaInvalida(f\'{origen}: salida "{r["salida"]}" no es {SALIDAS}\')\n    n = r[\'normalizacion\']\n    if n.get(\'metodo\') not in (\'percentil\', \'media_desv\', \'ninguna\'):\n        raise RecetaInvalida(f\'{origen}: normalización "{n.get("metodo")}" desconocida\')\n    if n[\'metodo\'] != \'ninguna\':\n        for c in n.get(\'canales\', []):\n            if c not in n.get(\'por_canal\', {}):\n                raise RecetaInvalida(f\'{origen}: faltan las constantes de "{c}"\')\n    for c, k in r.get(\'escala_terreno\', {}).items():\n        if c not in TERRENO or not float(k) > 0:\n            raise RecetaInvalida(f\'{origen}: escala_terreno["{c}"] no es válida\')\n    if not 0 < float(r[\'umbral\']) < 1:\n        raise RecetaInvalida(f\'{origen}: el umbral tiene que estar entre 0 y 1\')\n\n\ndef bandas_s2(r):\n    """Bandas de Sentinel-2 que hay que pedir para esta receta."""\n    s = set()\n    for c in r[\'canales\']:\n        if c in INDICES:\n            s.update(INDICES[c])\n        elif c.startswith(\'B\'):\n            s.add(c)\n    return sorted(s)\n\n\ndef necesita_dem(r):\n    return any(c in TERRENO or c in DERIVADOS_TERRENO for c in r[\'canales\'])\n\n\ndef _indice(a, b):\n    a = a.astype(np.float32); b = b.astype(np.float32)\n    den = a + b\n    out = np.zeros_like(a)\n    np.divide(a - b, den, out=out, where=den != 0)\n    return np.clip(out, -1.0, 1.0)\n\n\ndef _curvatura_dem(elev):\n    """Curvatura laplaciana discreta del DEM normalizada en [-1, 1].\n    Positivo = ladera cóncava/convergente (acumula agua); negativo = cresta convexa."""\n    z = elev.astype(np.float32)\n    zp = np.pad(z, 1, mode=\'edge\')\n    lap = (zp[:-2, 1:-1] + zp[2:, 1:-1] + zp[1:-1, :-2] + zp[1:-1, 2:]) - 4.0 * z\n    return np.tanh(lap * 15.0).astype(np.float32)\n\n\ndef _tpi_dem(elev):\n    """Topographic Position Index (TPI) en ventana 5×5, acotado en [-1, 1].\n    Positivo = crestas/zonas altas locales; negativo = fondos de valle/quebradas."""\n    z = elev.astype(np.float32)\n    zp = np.pad(z, 2, mode=\'edge\')\n    acc = np.zeros_like(z)\n    for dy in range(5):\n        for dx in range(5):\n            acc += zp[dy:dy + z.shape[0], dx:dx + z.shape[1]]\n    media_local = acc / 25.0\n    return np.tanh((z - media_local) * 15.0).astype(np.float32)\n\n\ndef armar(fuente, valido, r):\n    """fuente: canal -> (H, W) en unidades de Landslide4Sense.\n    valido: (H, W) bool. Devuelve (C, H, W) float32 listo para el ONNX."""\n    n = r[\'normalizacion\']\n    a_normalizar = set(n.get(\'canales\', []))\n    capas = []\n    for c in r[\'canales\']:\n        if c in INDICES:\n            x = _indice(fuente[INDICES[c][0]], fuente[INDICES[c][1]])\n        elif c == \'CURVATURA\':\n            x = _curvatura_dem(fuente[\'elevacion\'])\n        elif c == \'TPI\':\n            x = _tpi_dem(fuente[\'elevacion\'])\n        else:\n            x = fuente[c].astype(np.float32)\n        if c in a_normalizar:\n            k = n[\'por_canal\'][c]\n            if n[\'metodo\'] == \'percentil\':\n                lo, hi = float(k[\'lo\']), float(k[\'hi\'])\n                x = np.clip((x - lo) / max(hi - lo, 1e-9), 0.0, 1.0)\n            elif n[\'metodo\'] == \'media_desv\':\n                x = (x - float(k[\'media\'])) / max(float(k[\'desv\']), 1e-9)\n        capas.append(np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0))\n    t = np.stack(capas).astype(np.float32)\n    return t * valido.astype(np.float32)[None]\n\n\n# ------------------------------------------------------------ dos adaptadores\ndef desde_l4s(h5, r):\n    """Un parche de Landslide4Sense, (128, 128, 14) o (14, 128, 128), ya viene en\n    sus unidades. Solo hay que tomar los canales."""\n    a = np.asarray(h5, np.float32)\n    if a.shape[-1] == 14:\n        a = a.transpose(2, 0, 1)\n    valido = np.isfinite(a).all(axis=0)\n    a = np.nan_to_num(a, nan=0.0)\n    necesarios = set()\n    for c in r[\'canales\']:\n        if c in DERIVADOS_TERRENO:\n            necesarios.add(\'elevacion\')\n        else:\n            necesarios.update(INDICES.get(c, (c,)))\n    fuente = {c: a[IDX_L4S[c]] for c in necesarios}\n    return armar(fuente, valido, r), valido\n\n\ndef desde_l2a(bandas_dn, terreno, valido, r, escala=None):\n    """Bandas de Sentinel-2 L2A en niveles digitales (ya sin el desplazamiento de\n    la baseline 04.00), y pendiente en grados y elevación en metros. Se llevan a\n    unidades de Landslide4Sense dividiendo cada banda por su media de escena,\n    que es como se construyó Landslide4Sense."""\n    escala = escala or r.get(\'escala_bandas\') or config.ESCALA_L4S\n    if not escala:\n        raise RecetaInvalida(\'no hay escala de bandas: falta escala_laconvencion.json\')\n    fuente = {}\n    for b, x in bandas_dn.items():\n        if b not in escala:\n            raise RecetaInvalida(f\'no hay constante de escala para {b}\')\n        fuente[b] = x.astype(np.float32) / float(escala[b])\n    # pendiente en grados y elevación en metros, a unidades de Landslide4Sense;\n    # si la receta no trae sus constantes valen las de Landslide4Sense\n    k_terreno = dict(config.ESCALA_TERRENO_L4S, **r.get(\'escala_terreno\', {}))\n    for c, x in (terreno or {}).items():\n        fuente[c] = x.astype(np.float32) / float(k_terreno[c])\n    return armar(fuente, valido, r)\n\n\ndef probabilidad(salida, tipo):\n    """(N, C, H, W) -> (N, H, W) en [0, 1] para la clase deslizamiento."""\n    z = np.asarray(salida, np.float32)\n    if tipo == \'sigmoide\':\n        if z.shape[1] != 1:\n            raise RecetaInvalida(f\'la receta dice sigmoide y el modelo da {z.shape[1]} canales\')\n        return 1.0 / (1.0 + np.exp(-np.clip(z[:, 0], -60, 60)))\n    if z.shape[1] != 2:\n        raise RecetaInvalida(f\'la receta dice softmax y el modelo da {z.shape[1]} canales\')\n    z = z - z.max(axis=1, keepdims=True)\n    e = np.exp(z)\n    return e[:, 1] / e.sum(axis=1)\n'
ARCHIVOS['nucleo/modelos.py'] = '"""\nCarga de modelos ONNX con su receta, y la comprobación de que están entrenados.\n\nLos ONNX que había en el repositorio corrían sin error y no detectaban nada: sus\npesos eran los de inicialización de PyTorch, porque al exportar no se cargó el\ncheckpoint. Un fallo así no se ve en la forma del grafo ni en un error de\nejecución, solo en los pesos. Por eso cada modelo pasa por `revisar_pesos` antes\nde usarse: en una capa recién inicializada, el peso máximo es exactamente\n1/sqrt(fan_in), la media de |w| es la mitad y la desviación la de una uniforme.\nUna capa entrenada se aleja de eso.\n"""\nimport glob\nimport json\nimport hashlib\nimport os\nimport threading\n\nimport numpy as np\n\nfrom . import config, recetas\n\nCABECERA_LFS = b\'version https://git-lfs\'\n_LOCK = threading.Lock()\n_SESIONES = {}\n_FICHAS = {}\n_RUTA_SESION = {}     # nombre -> ONNX con el que se abre la sesión\n\n\n# ------------------------------------------------------------------ archivos\ndef es_puntero_lfs(ruta):\n    try:\n        with open(ruta, \'rb\') as f:\n            return f.read(len(CABECERA_LFS)) == CABECERA_LFS\n    except OSError:\n        return False\n\n\ndef sha256(ruta):\n    h = hashlib.sha256()\n    with open(ruta, \'rb\') as f:\n        for bloque in iter(lambda: f.read(1 << 20), b\'\'):\n            h.update(bloque)\n    return h.hexdigest()\n\n\ndef _asegurar_archivo(ruta):\n    """Si falta o es un puntero de Git LFS, lo baja de URL_PESOS_BASE."""\n    if os.path.exists(ruta) and not es_puntero_lfs(ruta):\n        return\n    if not config.URL_PESOS_BASE:\n        raise FileNotFoundError(\n            f\'{os.path.basename(ruta)} falta o llegó como puntero de Git LFS, y no hay \'\n            \'URL_PESOS_BASE para bajarlo\')\n    import httpx\n    url = f\'{config.URL_PESOS_BASE.rstrip("/")}/{os.path.basename(ruta)}\'\n    print(f\'[pesos] bajando {url}\', flush=True)\n    tmp = ruta + \'.parcial\'\n    with httpx.stream(\'GET\', url, follow_redirects=True, timeout=600) as r:\n        r.raise_for_status()\n        with open(tmp, \'wb\') as f:\n            for trozo in r.iter_bytes(1 << 20):\n                f.write(trozo)\n    if es_puntero_lfs(tmp):\n        os.remove(tmp)\n        raise RuntimeError(f\'{url} devolvió otro puntero LFS, no el archivo\')\n    os.replace(tmp, ruta)\n\n\n# --------------------------------------------------------- prueba de los pesos\ndef _parece_inicializacion(w):\n    """¿Esta capa conv tiene la forma exacta de una inicialización y no de un\n    peso entrenado? Se reconocen las de PyTorch por defecto, Kaiming y Xavier,\n    uniformes o normales, que son las que usan torchvision y\n    segmentation_models_pytorch. Un peso entrenado tiene colas y no calza con\n    ninguna a la vez en forma y en escala."""\n    w = np.asarray(w, np.float64)\n    fi = w.shape[1] * w.shape[2] * w.shape[3]\n    fo = w.shape[0] * w.shape[2] * w.shape[3]\n    c = np.abs(w).max()\n    s = w.std()\n    if c == 0 or s == 0:\n        return False\n    cotas = (1 / np.sqrt(fi), np.sqrt(3 / fi), np.sqrt(6 / fi), np.sqrt(6 / (fi + fo)),\n             np.sqrt(3 / fo), np.sqrt(6 / fo))\n    uniforme = (0.47 < np.abs(w).mean() / c < 0.53 and 0.95 < s / (c / np.sqrt(3)) < 1.05\n                and any(abs(c / k - 1) < 0.03 for k in cotas))\n    if uniforme:\n        return True\n    z = (w - w.mean()) / s\n    exceso_curtosis = float((z ** 4).mean() - 3)\n    desvs = (np.sqrt(2 / fi), np.sqrt(2 / fo), np.sqrt(1 / fi), np.sqrt(2 / (fi + fo)))\n    normal = (abs(w.mean()) < 0.05 * s and abs(exceso_curtosis) < 0.15\n              and any(abs(s / k - 1) < 0.02 for k in desvs))\n    return bool(normal)\n\n\ndef _forma(valor):\n    return [d.dim_value if d.HasField(\'dim_value\') else \'N\'\n            for d in valor.type.tensor_type.shape.dim]\n\n\ndef revisar_pesos(ruta_onnx, copia_aparte=None):\n    """Cuántas capas conv tienen pesos indistinguibles de la inicialización\n    uniforme de PyTorch, U(-1/sqrt(fan_in), +1/sqrt(fan_in)). De paso devuelve\n    la forma de la entrada y de la salida, para no abrir una sesión solo para eso."""\n    import onnx\n    from onnx import numpy_helper\n    m = onnx.load(ruta_onnx, load_external_data=True)\n    pesos = {t.name: t for t in m.graph.initializer}\n    nombres_init = set(pesos)\n    entradas = [i for i in m.graph.input if i.name not in nombres_init]\n    forma_in = _forma(entradas[0]) if entradas else []\n    forma_out = _forma(m.graph.output[0]) if m.graph.output else []\n    marcas = []                      # en el orden del grafo\n    for nodo in m.graph.node:\n        if nodo.op_type != \'Conv\' or len(nodo.input) < 2 or nodo.input[1] not in pesos:\n            continue\n        t = pesos[nodo.input[1]]\n        if len(t.dims) != 4 or int(np.prod(t.dims)) < 256:\n            continue\n        marcas.append(_parece_inicializacion(numpy_helper.to_array(t)))\n    del pesos\n    ruta_sesion = _guardar_pesos_aparte(m, ruta_onnx, copia_aparte) if copia_aparte else ruta_onnx\n    del m\n    liberar_memoria()\n    convs, en_init = len(marcas), sum(marcas)\n    frac = en_init / convs if convs else 0.0\n    cabeza = any(marcas[-3:])\n    return {\'capas_conv\': convs, \'capas_en_inicializacion\': en_init,\n            \'fraccion\': round(frac, 3), \'cabeza_sin_entrenar\': cabeza,\n            # la cabeza es la que decide la clase: si está en la inicialización el\n            # modelo no aprendió a segmentar, aunque el codificador traiga ImageNet\n            \'sin_entrenar\': frac >= 0.25 or cabeza,\n            \'_forma_entrada\': forma_in, \'_forma_salida\': forma_out,\n            \'_ruta_sesion\': ruta_sesion}\n\n\ndef _guardar_pesos_aparte(m, ruta_onnx, destino):\n    """Copia del ONNX con los pesos en un .onnx.data al lado. Si algo falla se usa\n    el original: es una optimización de memoria, no un requisito."""\n    import onnx\n    try:\n        if os.path.getsize(ruta_onnx) < 8 * 1024 * 1024:\n            return ruta_onnx                  # modelos chicos: no vale la pena\n        os.makedirs(destino, exist_ok=True)\n        base = os.path.splitext(os.path.basename(ruta_onnx))[0]\n        salida = os.path.join(destino, base + \'.onnx\')\n        datos_ext = salida + \'.data\'\n        esperado = sum(len(t.raw_data) for t in m.graph.initializer)\n        # Si ya existe una copia íntegra y actualizada, se reutiliza sin borrarla ni reescribirla:\n        # evita que varios procesos en paralelo (descargar_desde_puntos.py --paralelo) corrompan\n        # o trunquen .onnx.data mientras otro proceso está corriendo inferencia (causa del fallo en 012 y 015).\n        if (os.path.exists(salida) and os.path.exists(datos_ext)\n                and os.path.getmtime(salida) >= os.path.getmtime(ruta_onnx)\n                and esperado <= os.path.getsize(datos_ext) <= int(1.05 * esperado + 1024 * 1024)):\n            return salida\n        # onnx agrega al final de un .data existente en vez de reemplazarlo: sin\n        # borrarlo antes, cada arranque lo haría crecer otros 90 MB\n        for viejo in (salida, datos_ext):\n            if os.path.exists(viejo):\n                os.remove(viejo)\n        onnx.save_model(m, salida, save_as_external_data=True, all_tensors_to_one_file=True,\n                        location=os.path.basename(datos_ext), size_threshold=1024)\n        if os.path.getsize(datos_ext) > 1.05 * esperado + 1024 * 1024:\n            raise RuntimeError(f\'la copia de pesos pesa {os.path.getsize(datos_ext)} bytes\')\n        return salida\n    except Exception as e:\n        print(f\'[modelos] sin copia con pesos aparte ({type(e).__name__}: {e}); \'\n              \'se usa el ONNX original\', flush=True)\n        return ruta_onnx\n\n\ndef liberar_memoria():\n    """Devuelve al sistema la memoria que Python ya soltó. Sin esto, glibc la\n    retiene y el proceso parece ocupar más de lo que usa."""\n    import gc\n    gc.collect()\n    try:\n        import ctypes\n        ctypes.CDLL(\'libc.so.6\').malloc_trim(0)\n    except Exception:\n        pass\n\n\n# ----------------------------------------------------------------- catálogo\ndef _leer_ensamble(ruta):\n    """Un .json con "tipo": "ensamble" no apunta a un ONNX: promedia las\n    probabilidades de los modelos que nombra en "miembros"."""\n    with open(ruta, encoding=\'utf-8\') as f:\n        r = json.load(f)\n    if r.get(\'tipo\') != \'ensamble\':\n        return None\n    if not isinstance(r.get(\'miembros\'), list) or len(r[\'miembros\']) < 2:\n        raise ValueError(\'un ensamble necesita al menos dos "miembros"\')\n    r.setdefault(\'nombre\', os.path.splitext(os.path.basename(ruta))[0])\n    r[\'umbral\'] = float(r.get(\'umbral\', 0.5))\n    return r\n\n\ndef es_ensamble(r):\n    return isinstance(r, dict) and r.get(\'tipo\') == \'ensamble\'\n\n\ndef catalogo():\n    """nombre -> receta, leyendo modelos/*.json."""\n    out = {}\n    for ruta in sorted(glob.glob(os.path.join(config.CARPETA_MODELOS, \'*.json\'))):\n        try:\n            ens = _leer_ensamble(ruta)\n            if ens is not None:\n                out[ens[\'nombre\']] = ens\n                continue\n            r = recetas.leer(ruta)\n        except Exception as e:\n            print(f\'[modelos] receta ignorada {os.path.basename(ruta)}: {e}\', flush=True)\n            continue\n        r[\'_ruta_onnx\'] = os.path.join(config.CARPETA_MODELOS, r[\'onnx\'])\n        out[r[\'nombre\']] = r\n    return out\n\n\ndef ficha(nombre):\n    """Lo que /modelos publica de cada modelo. Se calcula una vez."""\n    if nombre in _FICHAS:\n        return _FICHAS[nombre]\n    cat = catalogo()\n    if nombre not in cat:\n        raise KeyError(nombre)\n    r = cat[nombre]\n    if es_ensamble(r):\n        return _ficha_ensamble(nombre, r, cat)\n    f = {\'nombre\': nombre, \'etiqueta\': r.get(\'etiqueta\', nombre),\n         \'autor\': r.get(\'autor\'), \'canales\': r[\'canales\'], \'lado_px\': r[\'lado_px\'],\n         \'salida\': r[\'salida\'], \'umbral\': r[\'umbral\'],\n         \'necesita_dem\': recetas.necesita_dem(r),\n         \'entrenamiento\': r.get(\'entrenamiento\'),\n         \'verificacion_l4s\': r.get(\'verificacion_l4s\')}\n    try:\n        _asegurar_archivo(r[\'_ruta_onnx\'])\n        # La revisión carga el ONNX entero en memoria un momento; se hace antes de\n        # cualquier sesión para que los dos picos no se sumen.\n        pesos = revisar_pesos(r[\'_ruta_onnx\'], config.CARPETA_COPIAS or None)\n        _RUTA_SESION[nombre] = pesos.pop(\'_ruta_sesion\')\n        forma_in = pesos.pop(\'_forma_entrada\')\n        forma_out = pesos.pop(\'_forma_salida\')\n        f[\'entrada\'] = {\'forma\': forma_in}\n        f[\'salida_forma\'] = forma_out\n        problemas = []\n        if len(forma_in) != 4:\n            problemas.append(f\'el ONNX tiene una entrada de forma {forma_in}; se esperaba N×C×H×W\')\n        else:\n            c, h, w = forma_in[1], forma_in[2], forma_in[3]\n            if c != len(r[\'canales\']):\n                problemas.append(f\'el ONNX pide {c} canales y la receta da {len(r["canales"])}\')\n            if (h, w) != (r[\'lado_px\'], r[\'lado_px\']):\n                problemas.append(f\'el ONNX pide {h}x{w} y la receta dice {r["lado_px"]}\')\n        f[\'pesos\'] = pesos\n        f[\'sha256\'] = sha256(r[\'_ruta_onnx\'])\n        if pesos[\'sin_entrenar\']:\n            f[\'estado\'] = \'sin_entrenar\'\n            f[\'detalle\'] = (f\'{pesos["capas_en_inicializacion"]} de {pesos["capas_conv"]} \'\n                            \'capas tienen los pesos de inicialización: este ONNX se \'\n                            \'exportó sin cargar el entrenamiento y no detecta nada.\')\n        elif problemas:\n            f[\'estado\'] = \'error\'\n            f[\'detalle\'] = \'; \'.join(problemas)\n        else:\n            f[\'estado\'] = \'listo\'\n    except Exception as e:\n        f[\'estado\'] = \'error\'\n        f[\'detalle\'] = f\'{type(e).__name__}: {e}\'\n    _FICHAS[nombre] = f\n    return f\n\n\ndef _ficha_ensamble(nombre, r, cat):\n    """Listo si todos sus miembros están listos y comparten canales y tamaño de parche."""\n    faltan = [m for m in r[\'miembros\'] if m not in cat or es_ensamble(cat[m])]\n    f = {\'nombre\': nombre, \'tipo\': \'ensamble\', \'etiqueta\': r.get(\'etiqueta\', nombre),\n         \'autor\': r.get(\'autor\'), \'miembros\': r[\'miembros\'], \'umbral\': r[\'umbral\'],\n         \'umbral_sensible\': r.get(\'umbral_sensible\'), \'salida\': \'promedio de probabilidades\',\n         \'entrenamiento\': r.get(\'entrenamiento\'), \'verificacion_l4s\': r.get(\'verificacion_l4s\')}\n    if faltan:\n        f.update(estado=\'error\', detalle=f\'faltan los modelos {faltan}\', canales=[], lado_px=128,\n                 necesita_dem=False)\n        _FICHAS[nombre] = f\n        return f\n    fm = [ficha(m) for m in r[\'miembros\']]\n    f[\'canales\'] = fm[0][\'canales\']; f[\'lado_px\'] = fm[0][\'lado_px\']\n    f[\'necesita_dem\'] = any(x[\'necesita_dem\'] for x in fm)\n    f[\'sha256\'] = {m: x.get(\'sha256\') for m, x in zip(r[\'miembros\'], fm)}\n    malos = [f"{x[\'etiqueta\']}: {x[\'estado\']}" for x in fm if x[\'estado\'] != \'listo\']\n    distintos = len({x[\'lado_px\'] for x in fm}) > 1 or len({tuple(x[\'canales\']) for x in fm}) > 1\n    if malos:\n        f.update(estado=\'error\', detalle=\'; \'.join(malos))\n    elif distintos:\n        f.update(estado=\'error\', detalle=\'los miembros no usan los mismos canales y tamaño de parche\')\n    else:\n        f[\'estado\'] = \'listo\'\n    _FICHAS[nombre] = f\n    return f\n\n\ndef _crear_sesion(ruta):\n    import onnxruntime as ort\n    so = ort.SessionOptions()\n    so.intra_op_num_threads = config.HILOS_ONNX\n    so.inter_op_num_threads = 1\n    so.enable_cpu_mem_arena = False\n    so.graph_optimization_level = {\n        \'basica\': ort.GraphOptimizationLevel.ORT_ENABLE_BASIC,\n        \'extendida\': ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED,\n        \'todas\': ort.GraphOptimizationLevel.ORT_ENABLE_ALL,\n    }.get(config.NIVEL_OPT_ONNX, ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED)\n    return ort.InferenceSession(ruta, so, providers=[\'CPUExecutionProvider\'])\n\n\ndef receta_lista(nombre):\n    """La receta de un modelo que puede usarse, sin cargar la sesión; o una\n    excepción que explica por qué no. Sirve para rechazar antes de bajar imágenes."""\n    cat = catalogo()\n    if nombre not in cat:\n        raise KeyError(f\'modelo "{nombre}" no existe. Disponibles: {sorted(cat)}\')\n    f = ficha(nombre)\n    if f[\'estado\'] == \'sin_entrenar\' and not config.PERMITIR_SIN_ENTRENAR:\n        raise PermissionError(f[\'detalle\'])\n    if f[\'estado\'] == \'error\':\n        raise RuntimeError(f[\'detalle\'])\n    if es_ensamble(cat[nombre]):\n        # para planificar y armar canales sirve la receta del primer miembro\n        return dict(receta_lista(cat[nombre][\'miembros\'][0]), _ensamble=cat[nombre])\n    return cat[nombre]\n\n\ndef obtener(nombre):\n    """(sesión, receta) lista para inferir, o una excepción que explica por qué no."""\n    receta = receta_lista(nombre)\n    with _LOCK:\n        if nombre not in _SESIONES:\n            if config.UN_MODELO_A_LA_VEZ and _SESIONES:\n                _SESIONES.clear()\n                liberar_memoria()\n            _SESIONES[nombre] = _crear_sesion(_RUTA_SESION.get(nombre, receta[\'_ruta_onnx\']))\n        return _SESIONES[nombre], receta\n\n\ndef soltar(nombre=None):\n    """Descarga la sesión (o todas) para que la memoria quede libre entre trabajos.\n    Volver a crearla cuesta menos de un segundo; tenerla cargada mientras se bajan\n    las imágenes cuesta más de 100 MB en un plan de 512."""\n    with _LOCK:\n        if nombre is None:\n            _SESIONES.clear()\n        else:\n            _SESIONES.pop(nombre, None)\n    liberar_memoria()\n\n\ndef cargados():\n    return sorted(_SESIONES)\n'
ARCHIVOS['escala_laconvencion.json'] = '{\n "metodo": "media por banda de reflectancia L2A (DN, tras quitar el desplazamiento de 1000 de la baseline 04.00) sobre pixeles sin nube de la provincia",\n "anio": 2024,\n "temporada": "junio-septiembre",\n "lado_tesela_px": 256,\n "puntos_usados": 14,\n "pixeles": 2546624,\n "media_dn": {\n  "B01": 381.62,\n  "B02": 442.57,\n  "B03": 576.95,\n  "B04": 465.56,\n  "B08": 2747.8\n },\n "desv_dn": {\n  "B01": 422.39,\n  "B02": 460.82,\n  "B03": 449.89,\n  "B04": 474.42,\n  "B08": 839.17\n },\n "p2_p98_dn": {\n  "B01": [\n   44.0,\n   1173.0\n  ],\n  "B02": [\n   88.0,\n   1220.0\n  ],\n  "B03": [\n   185.0,\n   1298.0\n  ],\n  "B04": [\n   110.0,\n   1354.0\n  ],\n  "B08": [\n   1134.0,\n   4628.0\n  ]\n },\n "detalle": [\n  {\n   "lat": -11.4301,\n   "lon": -72.7256,\n   "px": 196544,\n   "escenas": [\n    "2024-08-14",\n    "2024-09-03",\n    "2024-09-23"\n   ]\n  },\n  {\n   "lat": -12.9421,\n   "lon": -72.4233,\n   "px": 189104,\n   "escenas": [\n    "2024-07-17",\n    "2024-06-22",\n    "2024-06-02"\n   ]\n  },\n  {\n   "lat": -12.396,\n   "lon": -72.3804,\n   "px": 165642,\n   "escenas": [\n    "2024-08-14",\n    "2024-09-23",\n    "2024-06-10"\n   ]\n  },\n  {\n   "lat": -12.8224,\n   "lon": -73.3721,\n   "px": 196198,\n   "escenas": [\n    "2024-06-10",\n    "2024-08-14",\n    "2024-07-20"\n   ]\n  },\n  {\n   "lat": -12.4474,\n   "lon": -73.4688,\n   "px": 156404,\n   "escenas": [\n    "2024-06-10",\n    "2024-08-14",\n    "2024-09-23"\n   ]\n  },\n  {\n   "lat": -12.2035,\n   "lon": -72.9676,\n   "px": 169860,\n   "escenas": [\n    "2024-08-14",\n    "2024-09-23",\n    "2024-06-10"\n   ]\n  },\n  {\n   "lat": -13.35,\n   "lon": -72.7508,\n   "px": 192675,\n   "escenas": [\n    "2024-06-10",\n    "2024-08-14",\n    "2024-07-25"\n   ]\n  },\n  {\n   "lat": -12.2903,\n   "lon": -73.9088,\n   "px": 196124,\n   "escenas": [\n    "2024-06-10",\n    "2024-08-14",\n    "2024-09-23"\n   ]\n  },\n  {\n   "lat": -11.3852,\n   "lon": -73.0446,\n   "px": 196608,\n   "escenas": [\n    "2024-08-14",\n    "2024-09-03",\n    "2024-09-23"\n   ]\n  },\n  {\n   "lat": -12.2921,\n   "lon": -72.7173,\n   "px": 196608,\n   "escenas": [\n    "2024-08-14",\n    "2024-09-23",\n    "2024-06-10"\n   ]\n  },\n  {\n   "lat": -12.892,\n   "lon": -72.983,\n   "px": 115560,\n   "escenas": [\n    "2024-06-10",\n    "2024-08-14",\n    "2024-07-25"\n   ]\n  },\n  {\n   "lat": -12.9975,\n   "lon": -72.5913,\n   "px": 183254,\n   "escenas": [\n    "2024-06-10",\n    "2024-08-14",\n    "2024-07-25"\n   ]\n  },\n  {\n   "lat": -13.1013,\n   "lon": -72.3142,\n   "px": 195435,\n   "escenas": [\n    "2024-07-17",\n    "2024-06-22",\n    "2024-06-02"\n   ]\n  },\n  {\n   "lat": -11.5427,\n   "lon": -72.9571,\n   "px": 196608,\n   "escenas": [\n    "2024-08-14",\n    "2024-09-03",\n    "2024-09-23"\n   ]\n  }\n ]\n}'
ARCHIVOS['herramientas/entrenar.py'] = '"""\nEntrena U-Net o DeepLabV3+ con Landslide4Sense, con las entradas que pide el\nproyecto: Sentinel-2 B1, B2, B3, B4, pendiente (B13) y DEM (B14).\n\nQué hace, en orden:\n\n  1. Lee los h5 de Landslide4Sense (128 x 128 x 14) de los tres conjuntos\n     oficiales: entrenamiento (3799), validación (245) y prueba (800).\n  2. Calcula la media y la desviación de cada canal sobre el entrenamiento.\n  3. Arma cada parche con `recetas.desde_l4s`, la MISMA función que usa el\n     servidor en Render. Entrenamiento y detección normalizan igual por\n     construcción, no por cuidado.\n  4. Entrena con BCE + Dice, AdamW, coseno y giros/espejos (como la Fase 4).\n     Guarda la época con mejor IoU en validación.\n     Opciones para deslizamientos pequeños y zonas verdes (versión 2, apagadas por\n     defecto, así que sin ellas el entrenamiento es idéntico al original):\n       --peso-tamano    más peso en la pérdida a los deslizamientos pequeños\n       --peso-lovasz    suma la pérdida Lovász, que optimiza el IoU directamente\n       --copiar-pegar   pega deslizamientos pequeños sobre laderas verdes y empinadas\n       --sobremuestreo  muestra más seguido los parches con deslizamientos pequeños\n       --brillo         varía brillo y contraste de B1 a B4 (no del relieve)\n       --ema            elige y guarda un promedio móvil de los pesos, más estable\n       --seleccion iou_max  elige la época por su mejor IoU en cualquier umbral\n       --desde          continúa desde un punto de control ya entrenado\n     Opciones para un entrenamiento largo (versión 3, también apagadas por defecto):\n       --calentamiento  épocas con la tasa de aprendizaje subiendo de a poco al inicio\n       --lr-min         tasa mínima al final del coseno (en vez de llegar a cero)\n       --recorte-grad   recorta gradientes muy grandes, para que un lote raro no\n                        desarme lo aprendido\n       --promediar K    guarda las K mejores épocas y al final prueba su promedio de\n                        pesos; se queda con el promedio si gana en validación\n       --reanudar       guarda el avance cada pocas épocas y, si Colab se corta,\n                        sigue desde ahí al volver a correr el mismo comando\n     Opciones para bajar las falsas alarmas (versión 4, apagadas por defecto):\n       --ohem F         la BCE solo cuenta la fracción F de píxeles con más error\n                        (ejemplos difíciles), sobre todo falsas alarmas y bordes\n       --indices-verdor calcula dos índices de verdor de B2, B3 y B4 (GRVI y ExG)\n                        dentro del modelo; van dentro del ONNX y el servidor sigue\n                        enviando los mismos 6 canales\n  5. Elige el umbral que maximiza el IoU en validación y recién entonces mide\n     una sola vez en prueba. Mide además con TTA (promedio de la imagen y sus\n     espejos), que exportar_onnx.py puede meter dentro del ONNX con --tta.\n  6. Deja tres archivos con la misma firma, que exportar_onnx.py lee sin\n     opciones extra:\n        mejor_<arq>_b1b4b13b14_<firma>.pt\n        stats_norm_<firma>.json\n        resultados_<arq>_b1b4b13b14_<firma>.json\n\nUso (Colab con GPU):\n  python3 entrenar.py --l4s /content/l4s --arquitectura unet --salida /content/drive/MyDrive/entrenamiento\n  python3 entrenar.py --l4s /content/l4s --arquitectura deeplabv3plus --salida ...\n\nRequiere: torch, segmentation-models-pytorch, h5py, numpy.\n"""\nimport argparse\nimport csv\nimport glob\nimport hashlib\nimport json\nimport os\nimport re\nimport sys\nimport time\n\nimport numpy as np\n\nAQUI = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, os.path.join(AQUI, \'..\', \'backend\'))\nsys.path.insert(0, os.path.join(AQUI, \'..\'))          # repositorio v2: nucleo/ está en la raíz\n\nCANALES_PROYECTO = [\'B01\', \'B02\', \'B03\', \'B04\', \'pendiente\', \'elevacion\']\nETIQUETA_CANALES = \'b1b4b13b14\'\n\n\n# ------------------------------------------------------------------- datos\ndef pares(raiz, split, limite=None):\n    """(imagen, máscara) emparejados por nombre: images/<split>/image_12.h5 <->\n    annotations/<split>/mask_12.h5. Solo esos nombres: el espejo de Hugging Face\n    trae en annotations/train cinco archivos vacíos llamados image_*.h5 que no son\n    máscaras. Un par con algún archivo vacío o roto se salta y se avisa."""\n    import h5py\n    num = lambda f: int(re.search(r\'_(\\d+)\\.h5$\', os.path.basename(f)).group(1))\n    imgs = {num(f): f for f in glob.glob(os.path.join(raiz, \'images\', split, \'image_*.h5\'))}\n    msks = {num(f): f for f in glob.glob(os.path.join(raiz, \'annotations\', split, \'mask_*.h5\'))}\n    comunes = sorted(set(imgs) & set(msks))\n    if not comunes:\n        raise SystemExit(f\'no hay pares imagen/máscara en {raiz}/images/{split}\')\n    sin_par = len(set(imgs) ^ set(msks))\n    if sin_par:\n        print(f\'  aviso: {sin_par} archivo(s) de {split} sin su par; no se usan\')\n    if limite:\n        comunes = comunes[:limite]\n    buenos, malos = [], []\n    for k in comunes:\n        try:\n            with h5py.File(imgs[k], \'r\') as f:\n                ok_i = len(f.keys()) > 0 and f[list(f.keys())[0]].shape[-1] == 14\n            with h5py.File(msks[k], \'r\') as f:\n                ok_m = len(f.keys()) > 0 and f[list(f.keys())[0]].size == 128 * 128\n        except OSError:\n            ok_i = ok_m = False\n        (buenos if ok_i and ok_m else malos).append(k)\n    if malos:\n        print(f\'  aviso: {len(malos)} par(es) de {split} con archivos vacíos o rotos; se \'\n              f\'saltan: {[os.path.basename(imgs[k]) for k in malos[:10]]}\')\n    return [(imgs[k], msks[k]) for k in buenos]\n\n\ndef leer_h5(ruta):\n    import h5py\n    with h5py.File(ruta, \'r\') as f:\n        return np.asarray(f[list(f.keys())[0]])\n\n\ndef estadisticas(lista, canales):\n    """Media y desviación de cada canal sobre los píxeles válidos del entrenamiento,\n    en unidades de Landslide4Sense."""\n    from nucleo.recetas import IDX_L4S\n    idx = [IDX_L4S[c] for c in canales]\n    s = np.zeros(len(idx)); s2 = np.zeros(len(idx)); n = 0\n    muestras = [[] for _ in idx]\n    for img, _ in lista:\n        a = leer_h5(img).astype(np.float64)\n        if a.shape[-1] == 14:\n            a = a.transpose(2, 0, 1)\n        a = a[idx]\n        v = np.isfinite(a).all(axis=0)\n        x = a[:, v]\n        s += x.sum(axis=1); s2 += (x ** 2).sum(axis=1); n += x.shape[1]\n        for k in range(len(idx)):\n            muestras[k].append(x[k, ::37])\n    media = s / n\n    desv = np.sqrt(np.maximum(s2 / n - media ** 2, 1e-12))\n    # el rango que vio el modelo: el servidor avisa si un área cae fuera de él\n    rango = [np.percentile(np.concatenate(m), [0.1, 99.9]) for m in muestras]\n    return {c: {\'media\': round(float(m), 6), \'desv\': round(float(d), 6),\n                \'min\': round(float(r[0]), 6), \'max\': round(float(r[1]), 6)}\n            for c, m, d, r in zip(canales, media, desv, rango)}\n\n\ndef receta_de(canales, stats):\n    """La receta con la que se arma el tensor: la misma que irá junto al ONNX."""\n    return {\'nombre\': \'entrenamiento\', \'onnx\': \'-\', \'canales\': canales, \'lado_px\': 128,\n            \'salida\': \'sigmoide\', \'umbral\': 0.5,\n            \'normalizacion\': {\'metodo\': \'media_desv\',\n                              \'canales\': [c for c in canales if c in stats],\n                              \'por_canal\': stats}}\n\n\ndef cargar(lista, receta):\n    """Tensores (N, C, 128, 128) float32, máscaras (N, 128, 128) y válidos."""\n    from nucleo import recetas\n    X = np.zeros((len(lista), len(receta[\'canales\']), 128, 128), np.float32)\n    Y = np.zeros((len(lista), 128, 128), np.float32)\n    V = np.zeros((len(lista), 128, 128), np.float32)\n    for i, (img, msk) in enumerate(lista):\n        t, v = recetas.desde_l4s(leer_h5(img), receta)\n        X[i], V[i] = t, v\n        Y[i] = (np.asarray(leer_h5(msk)).squeeze() > 0)\n    return X, Y, V\n\n\n# ------------------------------------------------------------------ modelo\ndef perdida_mixta(logits, y, v, peso_dice=0.5, ohem=0.0):\n    import torch.nn.functional as F\n    import torch\n    bce = F.binary_cross_entropy_with_logits(logits, y, reduction=\'none\')\n    if ohem > 0:\n        # ejemplos difíciles: solo cuenta la fracción `ohem` de píxeles válidos con más error\n        # (casi siempre falsas alarmas y bordes), en vez del promedio de todos\n        l = (bce * v).flatten()\n        k = max(int(ohem * float((v > 0).sum())), 1)\n        bce = torch.topk(l, min(k, l.numel())).values.mean()\n    else:\n        bce = (bce * v).sum() / v.sum().clamp(min=1.0)\n    p = torch.sigmoid(logits) * v\n    yt = y * v\n    dice = 1.0 - (2.0 * (p * yt).sum() + 1.0) / (p.sum() + yt.sum() + 1.0)\n    return (1.0 - peso_dice) * bce + peso_dice * dice\n\n\ndef indices_verdor_np(X, canales, por_canal):\n    """GRVI = (G - R) / (G + R) y ExG = (2G - R - B) / (G + R + B), en unidades de\n    Landslide4Sense, a partir del tensor ya normalizado. Igual que ConIndices."""\n    def crudo(c):\n        return X[:, canales.index(c)] * por_canal[c][\'desv\'] + por_canal[c][\'media\']\n    g, r, b = crudo(\'B03\'), crudo(\'B04\'), crudo(\'B02\')\n    grvi = (g - r) / (np.abs(g + r) + 1e-3)\n    exg = (2 * g - r - b) / (np.abs(g + r + b) + 1e-3)\n    return np.clip(np.stack([grvi, exg], 1), -1, 1)\n\n\ndef modelo_con_indices(base, canales, por_canal=None, med_ind=None, des_ind=None):\n    """Envuelve el modelo: recibe los 6 canales de siempre, calcula dentro dos índices\n    de verdor de B2, B3 y B4 y le pasa 8 canales al modelo. Va dentro del ONNX, así\n    que el servidor no cambia. Sin por_canal (al exportar) las constantes llegan con\n    el state_dict."""\n    import torch\n    sel = [canales.index(c) for c in (\'B03\', \'B04\', \'B02\')]\n\n    class ConIndices(torch.nn.Module):\n        def __init__(self, m):\n            super().__init__()\n            self.base = m\n            if por_canal:\n                me = [por_canal[c][\'media\'] for c in (\'B03\', \'B04\', \'B02\')]\n                de = [por_canal[c][\'desv\'] for c in (\'B03\', \'B04\', \'B02\')]\n            else:\n                me, de = [0.0] * 3, [1.0] * 3\n            self.register_buffer(\'media\', torch.tensor(me, dtype=torch.float32).view(1, 3, 1, 1))\n            self.register_buffer(\'desv\', torch.tensor(de, dtype=torch.float32).view(1, 3, 1, 1))\n            self.register_buffer(\'media_ind\', torch.tensor(med_ind if med_ind is not None else [0.0, 0.0],\n                                                           dtype=torch.float32).view(1, 2, 1, 1))\n            self.register_buffer(\'desv_ind\', torch.tensor(des_ind if des_ind is not None else [1.0, 1.0],\n                                                          dtype=torch.float32).view(1, 2, 1, 1))\n\n        def forward(self, x):\n            rgb = x[:, sel] * self.desv + self.media\n            g, r, b = rgb[:, 0:1], rgb[:, 1:2], rgb[:, 2:3]\n            grvi = (g - r) / (torch.abs(g + r) + 1e-3)\n            exg = (2 * g - r - b) / (torch.abs(g + r + b) + 1e-3)\n            ind = torch.clamp(torch.cat([grvi, exg], 1), -1.0, 1.0)\n            ind = (ind - self.media_ind) / self.desv_ind\n            return self.base(torch.cat([x, ind], 1))\n\n    return ConIndices(base)\n\n\ndef mapa_pesos_tamano(Y, ref=50.0, maximo=5.0):\n    """Peso por píxel: 1 en el fondo y en deslizamientos grandes; hasta `maximo`\n    en los pequeños, según sqrt(ref / tamaño). Así perder uno chico cuesta más."""\n    from scipy import ndimage\n    W = np.ones_like(Y, np.float32)\n    for i in range(len(Y)):\n        lab, n = ndimage.label(Y[i] > 0)\n        if not n:\n            continue\n        tam = ndimage.sum(np.ones_like(lab), lab, range(1, n + 1))\n        w = np.clip(np.sqrt(ref / np.maximum(tam, 1)), 1.0, maximo).astype(np.float32)\n        W[i][lab > 0] = w[lab[lab > 0] - 1]\n    return W\n\n\ndef objetos(Y, max_px=None):\n    """Lista de deslizamientos (componentes conectados): (parche, filas, columnas, tamaño)."""\n    from scipy import ndimage\n    out = []\n    for i in range(len(Y)):\n        lab, n = ndimage.label(Y[i] > 0)\n        for j, sl in enumerate(ndimage.find_objects(lab), 1):\n            m = lab[sl] == j\n            if max_px is None or m.sum() <= max_px:\n                out.append((i, sl, m, int(m.sum())))\n    return out\n\n\ndef recall_por_tamano(prob, Y, V, umbral, objs=None):\n    """Fracción de deslizamientos tocados por lo marcado, por tamaño (px a 10 m)."""\n    objs = objs if objs is not None else objetos(Y)\n    tramos = {\'<10\': (1, 10), \'10-50\': (10, 50), \'50-200\': (50, 200), \'>200\': (200, 10 ** 9)}\n    hit = {k: [0, 0] for k in tramos}\n    for i, sl, m, t in objs:\n        toca = bool(((prob[i][sl] >= umbral) & (V[i][sl] > 0) & m).any())\n        for k, (a, b) in tramos.items():\n            if a <= t < b:\n                hit[k][0] += toca; hit[k][1] += 1\n    return {k: round(h / max(n, 1), 4) for k, (h, n) in hit.items()}\n\n\nclass Pegador:\n    """Copia-pega: lleva deslizamientos pequeños de un parche a otro, solo en las\n    bandas espectrales y solo sobre píxeles empinados y verdes del destino, para que\n    el modelo vea más ejemplos de lo que hoy se le escapa. El relieve del destino no\n    se toca: el deslizamiento queda donde el terreno lo permite."""\n    def __init__(self, X, Y, receta, rng, max_px=50, pend_min=15.0):\n        st = receta[\'normalizacion\'][\'por_canal\']; can = receta[\'canales\']\n        self.esp = [k for k, c in enumerate(can) if c.startswith(\'B\')]\n        self.ip = can.index(\'pendiente\') if \'pendiente\' in can else None\n        self.ig, self.ir = (can.index(\'B03\') if \'B03\' in can else None), (can.index(\'B04\') if \'B04\' in can else None)\n        self.m = {k: st[c][\'media\'] for k, c in enumerate(can) if c in st}\n        self.s = {k: st[c][\'desv\'] for k, c in enumerate(can) if c in st}\n        self.X, self.Y, self.rng, self.pend_min = X, Y, rng, pend_min\n        self.objs = objetos(Y, max_px)\n\n    def crudo(self, x, k):\n        return x[k] * self.s[k] + self.m[k]\n\n    def aplicar(self, x, y, w, n=2):\n        if not self.objs:\n            return x, y, w\n        apto = y < 0.5\n        if self.ip is not None:\n            apto &= self.crudo(x, self.ip) * 20.292 >= self.pend_min\n        if self.ig is not None and self.ir is not None:\n            apto &= self.crudo(x, self.ig) >= self.crudo(x, self.ir)\n        cand = np.argwhere(apto)\n        if len(cand) == 0:\n            return x, y, w\n        for _ in range(n):\n            i, sl, m, t = self.objs[int(self.rng.integers(len(self.objs)))]\n            h, ww = m.shape\n            dst = None\n            for _intento in range(20):\n                # el centro del deslizamiento cae en un píxel apto, y todo él tiene que\n                # quedar sobre píxeles aptos (empinados, verdes y sin otro deslizamiento)\n                fc, cc = cand[int(self.rng.integers(len(cand)))]\n                f0 = int(np.clip(fc - h // 2, 0, 128 - h)); c0 = int(np.clip(cc - ww // 2, 0, 128 - ww))\n                prueba = (slice(f0, f0 + h), slice(c0, c0 + ww))\n                if apto[prueba][m].all():\n                    dst = prueba\n                    break\n            if dst is None:\n                continue\n            apto[dst][m] = False\n            for k in self.esp:\n                x[k][dst][m] = self.X[i, k][sl][m]\n            y[dst][m] = 1.0\n            w[dst][m] = np.clip(np.sqrt(50.0 / max(t, 1)), 1.0, 5.0)\n        return x, y, w\n\n\nclass Conteo:\n    """tp, fp, fn, tn globales (sobre píxeles) para varios umbrales a la vez."""\n    def __init__(self, umbrales):\n        self.u = np.asarray(umbrales, np.float32)\n        self.c = np.zeros((len(self.u), 4), np.int64)\n\n    def sumar(self, prob, y, v):\n        ok = v > 0\n        p, t = prob[ok], y[ok] > 0\n        for i, u in enumerate(self.u):\n            m = p >= u\n            self.c[i] += [np.sum(m & t), np.sum(m & ~t), np.sum(~m & t), np.sum(~m & ~t)]\n\n    def metricas(self, i):\n        tp, fp, fn, tn = (int(x) for x in self.c[i])\n        pr = tp / max(tp + fp, 1); rc = tp / max(tp + fn, 1)\n        return {\'umbral\': round(float(self.u[i]), 2), \'iou\': round(tp / max(tp + fp + fn, 1), 4),\n                \'f1\': round(2 * pr * rc / max(pr + rc, 1e-9), 4), \'precision\': round(pr, 4),\n                \'recall\': round(rc, 4), \'tp\': tp, \'fp\': fp, \'fn\': fn, \'tn\': tn}\n\n\ndef logits_tta(modelo, x, tta=1):\n    """Logits promediados sobre la imagen y sus espejos: 1 = sin TTA, 2 = más espejo\n    horizontal, 4 = más espejo vertical y ambos. Igual que el envoltorio del ONNX."""\n    import torch\n    vueltas = [(), (-1,), (-2,), (-2, -1)][:tta]\n    suma = None\n    for d in vueltas:\n        z = modelo(torch.flip(x, d) if d else x)\n        z = torch.flip(z, d) if d else z\n        suma = z if suma is None else suma + z\n    return suma / len(vueltas)\n\n\ndef predecir(modelo, X, disp, lote=32, tta=1):\n    import torch\n    modelo.eval()\n    with torch.no_grad():\n        return np.concatenate([torch.sigmoid(logits_tta(modelo, torch.from_numpy(X[i:i + lote]).to(disp), tta)[:, 0])\n                               .float().cpu().numpy() for i in range(0, len(X), lote)])\n\n\ndef evaluar(modelo, X, Y, V, umbrales, disp, lote=32, tta=1):\n    cont = Conteo(umbrales)\n    for i in range(0, len(X), lote):\n        prob = predecir(modelo, X[i:i + lote], disp, lote, tta)\n        cont.sumar(prob, Y[i:i + lote], V[i:i + lote])\n    return cont\n\n\ndef factor_lr(ep, total, calentamiento, lr_min_rel):\n    """Multiplicador de la tasa de aprendizaje en la época ep (desde 0): sube en línea\n    recta durante el calentamiento y después baja en coseno hasta lr_min_rel."""\n    import math\n    if ep < calentamiento:\n        return (ep + 1) / calentamiento\n    t = (ep - calentamiento) / max(total - calentamiento, 1)\n    return lr_min_rel + (1.0 - lr_min_rel) * 0.5 * (1.0 + math.cos(math.pi * min(t, 1.0)))\n\n\ndef promedio_de_pesos(estados):\n    """Promedio simple de varios state_dict (los enteros, como num_batches_tracked,\n    se toman del primero)."""\n    out = {}\n    for k, v in estados[0].items():\n        if v.dtype.is_floating_point:\n            out[k] = sum(e[k].float() for e in estados) / len(estados)\n            out[k] = out[k].to(v.dtype)\n        else:\n            out[k] = v.clone()\n    return out\n\n\ndef recalcular_bn(modelo, X, disp, lote=32):\n    """Las medias y varianzas de BatchNorm de un promedio de pesos no son las de\n    ninguna época: se vuelven a medir pasando el entrenamiento (sin aumentos)."""\n    import torch\n    bns = [m for m in modelo.modules() if isinstance(m, torch.nn.modules.batchnorm._BatchNorm)]\n    if not bns:\n        return\n    guardado = [m.momentum for m in bns]\n    for m in bns:\n        m.reset_running_stats(); m.momentum = None        # promedio acumulado exacto\n    modelo.train()\n    with torch.no_grad():\n        for i in range(0, len(X) - 1, lote):\n            if len(X[i:i + lote]) < 2:\n                continue\n            modelo(torch.from_numpy(X[i:i + lote]).to(disp))\n    for m, mo in zip(bns, guardado):\n        m.momentum = mo\n    modelo.eval()\n\n\ndef guardar_seguro(obj, ruta):\n    """Escribe primero a un temporal y lo renombra: si Colab se corta a mitad, el\n    archivo anterior sigue entero."""\n    import torch\n    tmp = ruta + \'.tmp\'\n    torch.save(obj, tmp)\n    os.replace(tmp, ruta)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--l4s\', required=True, help=\'carpeta con images/ y annotations/\')\n    ap.add_argument(\'--arquitectura\', required=True, choices=[\'unet\', \'deeplabv3plus\'])\n    ap.add_argument(\'--salida\', required=True)\n    ap.add_argument(\'--canales\', default=\',\'.join(CANALES_PROYECTO))\n    ap.add_argument(\'--encoder\', default=\'resnet34\')\n    ap.add_argument(\'--encoder-weights\', dest=\'encoder_weights\', default=\'imagenet\',\n                    help="\'imagenet\' o \'ninguno\'")\n    ap.add_argument(\'--epocas\', type=int, default=100)\n    ap.add_argument(\'--paciencia\', type=int, default=30)\n    ap.add_argument(\'--lote\', type=int, default=16)\n    ap.add_argument(\'--lr\', type=float, default=3e-4)\n    ap.add_argument(\'--weight-decay\', dest=\'wd\', type=float, default=1e-4)\n    ap.add_argument(\'--peso-dice\', dest=\'peso_dice\', type=float, default=0.5)\n    ap.add_argument(\'--semilla\', type=int, default=42)\n    ap.add_argument(\'--limite\', type=int, help=\'solo para probar: parches por conjunto\')\n    # versión 2: deslizamientos pequeños y zonas verdes (todo apagado por defecto)\n    ap.add_argument(\'--peso-tamano\', dest=\'peso_tamano\', type=float, default=0.0,\n                    help=\'peso máximo para deslizamientos pequeños (0 = apagado; sugerido 5)\')\n    ap.add_argument(\'--peso-lovasz\', dest=\'peso_lovasz\', type=float, default=0.0,\n                    help=\'peso de la pérdida Lovász que se suma (0 = apagado; sugerido 0.5)\')\n    ap.add_argument(\'--copiar-pegar\', dest=\'copiar_pegar\', type=float, default=0.0,\n                    help=\'probabilidad de pegar deslizamientos pequeños en cada parche (sugerido 0.3)\')\n    ap.add_argument(\'--sobremuestreo\', type=float, default=1.0,\n                    help=\'cuántas veces más se muestran los parches con deslizamientos pequeños\')\n    ap.add_argument(\'--brillo\', type=float, default=0.0,\n                    help=\'variación máxima de brillo y contraste en B1 a B4 (sugerido 0.1)\')\n    ap.add_argument(\'--ema\', type=float, default=0.0, help=\'decaimiento del promedio de pesos (sugerido 0.999)\')\n    ap.add_argument(\'--seleccion\', choices=[\'iou05\', \'iou_max\'], default=\'iou05\')\n    ap.add_argument(\'--desde\', help=\'punto de control .pt de entrenar.py para continuar desde él\')\n    # versión 3: entrenamiento largo (todo apagado por defecto)\n    ap.add_argument(\'--calentamiento\', type=int, default=0, help=\'épocas de calentamiento (sugerido 5)\')\n    ap.add_argument(\'--lr-min\', dest=\'lr_min\', type=float, default=0.0, help=\'tasa mínima al final (sugerido 1e-6)\')\n    ap.add_argument(\'--recorte-grad\', dest=\'recorte_grad\', type=float, default=0.0,\n                    help=\'norma máxima del gradiente (0 = apagado; sugerido 1.0)\')\n    ap.add_argument(\'--promediar\', type=int, default=0,\n                    help=\'promedia los pesos de las K mejores épocas (0 = apagado; sugerido 5)\')\n    ap.add_argument(\'--reanudar\', action=\'store_true\',\n                    help=\'guarda el avance y, si existe, continúa desde él\')\n    ap.add_argument(\'--guardar-cada\', dest=\'guardar_cada\', type=int, default=5,\n                    help=\'con --reanudar, cada cuántas épocas se guarda el avance\')\n    # versión 4: menos falsas alarmas y mejor en zonas verdes (apagado por defecto)\n    ap.add_argument(\'--ohem\', type=float, default=0.0,\n                    help=\'fracción de píxeles más difíciles que cuenta en la BCE (0 = todos; sugerido 0.25)\')\n    ap.add_argument(\'--indices-verdor\', dest=\'indices_verdor\', action=\'store_true\',\n                    help=\'calcula GRVI y ExG de B2, B3 y B4 dentro del modelo (van en el ONNX)\')\n    a = ap.parse_args()\n\n    import torch\n    import segmentation_models_pytorch as smp\n    from nucleo.recetas import ALIAS, IDX_L4S\n\n    canales = [ALIAS.get(c.strip(), c.strip()) for c in a.canales.split(\',\') if c.strip()]\n    malos = [c for c in canales if c not in IDX_L4S]\n    if malos:\n        raise SystemExit(f\'canales que no están en Landslide4Sense: {malos}\')\n    torch.manual_seed(a.semilla); np.random.seed(a.semilla)\n    rng = np.random.default_rng(a.semilla)\n    disp = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    amp = disp == \'cuda\'\n    os.makedirs(a.salida, exist_ok=True)\n\n    config = {\'dataset\': \'Landslide4Sense (ibm-nasa-geospatial/Landslide4sense, conjuntos oficiales)\',\n              \'canales\': canales, \'normalizacion\': \'media_desv\', \'encoder\': a.encoder,\n              \'encoder_weights\': a.encoder_weights, \'tam_parche\': 128, \'lote\': a.lote,\n              \'epocas\': a.epocas, \'paciencia\': a.paciencia, \'lr\': a.lr, \'weight_decay\': a.wd,\n              \'peso_dice\': a.peso_dice, \'semilla\': a.semilla,\n              \'particion\': \'oficial: entrenamiento, validación (época y umbral), prueba (una vez)\',\n              \'aumentos\': \'giros de 90° y espejo horizontal\', \'limite\': a.limite}\n    v2 = {k: getattr(a, k) for k, d in ((\'peso_tamano\', 0.0), (\'peso_lovasz\', 0.0), (\'copiar_pegar\', 0.0),\n                                         (\'sobremuestreo\', 1.0), (\'brillo\', 0.0), (\'ema\', 0.0),\n                                         (\'seleccion\', \'iou05\'), (\'desde\', None)) if getattr(a, k) != d}\n    if v2:\n        config[\'v2\'] = dict(v2, desde=os.path.basename(a.desde) if a.desde else None)\n    v3 = {k: getattr(a, k) for k, d in ((\'calentamiento\', 0), (\'lr_min\', 0.0), (\'recorte_grad\', 0.0),\n                                         (\'promediar\', 0)) if getattr(a, k) != d}\n    if v3:\n        config[\'v3\'] = v3\n    v4 = {k: getattr(a, k) for k, d in ((\'ohem\', 0.0), (\'indices_verdor\', False)) if getattr(a, k) != d}\n    if v4:\n        config[\'v4\'] = v4\n    if a.indices_verdor and not all(c in canales for c in (\'B02\', \'B03\', \'B04\')):\n        raise SystemExit(\'--indices-verdor necesita B02, B03 y B04 entre los canales\')\n    if a.indices_verdor and a.desde:\n        raise SystemExit(\'--indices-verdor cambia la entrada del modelo: no se puede continuar con --desde\')\n    firma = hashlib.sha1(json.dumps(dict({k: config[k] for k in config if k != \'epocas\'},\n                                         arquitectura=a.arquitectura),\n                                    sort_keys=True).encode()).hexdigest()[:12]\n    base = f\'{a.arquitectura}_{ETIQUETA_CANALES}_{firma}\'\n    print(f\'firma {firma} | {a.arquitectura} {a.encoder} | canales {canales} | {disp}\')\n    ruta_res = os.path.join(a.salida, f\'resultados_{base}.json\')\n    ruta_ultimo = os.path.join(a.salida, f\'ultimo_{base}.pt\')\n    if a.reanudar and os.path.exists(ruta_res) and not os.path.exists(ruta_ultimo):\n        print(f\'este entrenamiento ya terminó antes ({os.path.basename(ruta_res)}): no se repite\')\n        return\n\n    t0 = time.time()\n    tr, va, te = (pares(a.l4s, s, a.limite) for s in (\'train\', \'validation\', \'test\'))\n    print(f\'parches: entrenamiento {len(tr)}, validación {len(va)}, prueba {len(te)}\')\n    stats = estadisticas(tr, canales)\n    ruta_stats = os.path.join(a.salida, f\'stats_norm_{firma}.json\')\n    json.dump(stats, open(ruta_stats, \'w\', encoding=\'utf-8\'), indent=1)\n    print(\'media y desviación del entrenamiento:\', json.dumps(stats))\n    if \'elevacion\' in stats:\n        print(f"rango del DEM en el entrenamiento: {stats[\'elevacion\'][\'min\']*772.82:.0f} a "\n              f"{stats[\'elevacion\'][\'max\']*772.82:.0f} m; pendiente hasta "\n              f"{stats.get(\'pendiente\', {}).get(\'max\', 0)*20.292:.0f}°")\n    receta = receta_de(canales, stats)\n    Xtr, Ytr, Vtr = cargar(tr, receta)\n    Xva, Yva, Vva = cargar(va, receta)\n    Xte, Yte, Vte = cargar(te, receta)\n    print(f\'datos en memoria en {time.time()-t0:.0f} s; píxeles de deslizamiento en \'\n          f\'entrenamiento: {Ytr.mean()*100:.2f} %\')\n\n    Wtr = mapa_pesos_tamano(Ytr, maximo=a.peso_tamano) if a.peso_tamano > 1 else None\n    objs_va = objetos(Yva)\n    p_muestra = None\n    if a.sobremuestreo > 1:\n        chicos = {i for i, _, _, t in objetos(Ytr, 50)}\n        p_muestra = np.array([a.sobremuestreo if i in chicos else 1.0 for i in range(len(Xtr))])\n        p_muestra /= p_muestra.sum()\n        print(f\'sobremuestreo: {len(chicos)} parches con deslizamientos pequeños, x{a.sobremuestreo:g}\')\n    pegador = Pegador(Xtr, Ytr, receta, rng) if a.copiar_pegar > 0 else None\n    if pegador:\n        print(f\'copiar-pegar: {len(pegador.objs)} deslizamientos pequeños disponibles\')\n    esp = [k for k, c in enumerate(canales) if c.startswith(\'B\')]\n    lovasz = None\n    if a.peso_lovasz > 0:\n        lovasz = smp.losses.LovaszLoss(\'binary\', per_image=False, ignore_index=2, from_logits=True)\n\n    pesos = None if a.encoder_weights in (\'ninguno\', \'none\', \'\') else a.encoder_weights\n    clase = {\'unet\': smp.Unet, \'deeplabv3plus\': smp.DeepLabV3Plus}[a.arquitectura]\n    extra = 2 if a.indices_verdor else 0\n    modelo = clase(encoder_name=a.encoder, encoder_weights=pesos, in_channels=len(canales) + extra,\n                   classes=1)\n    if a.indices_verdor:\n        pc = receta[\'normalizacion\'][\'por_canal\']\n        ind = indices_verdor_np(Xtr[::5], canales, pc)\n        med_ind = [float(ind[:, k].mean()) for k in range(2)]\n        des_ind = [float(max(ind[:, k].std(), 1e-3)) for k in range(2)]\n        modelo = modelo_con_indices(modelo, canales, pc, med_ind, des_ind)\n        print(f\'índices de verdor dentro del modelo: GRVI media {med_ind[0]:.3f} desv {des_ind[0]:.3f}, \'\n              f\'ExG media {med_ind[1]:.3f} desv {des_ind[1]:.3f}\')\n    modelo = modelo.to(disp)\n    if a.desde:\n        ck0 = torch.load(a.desde, map_location=disp, weights_only=False)\n        if ck0.get(\'arquitectura\') not in (None, a.arquitectura):\n            raise SystemExit(f"--desde es de {ck0.get(\'arquitectura\')}, no de {a.arquitectura}")\n        c0 = (ck0.get(\'config\') or {}).get(\'canales\')\n        if c0 and list(c0) != canales:\n            raise SystemExit(f\'--desde usa los canales {c0}, no {canales}\')\n        modelo.load_state_dict(ck0[\'modelo\'], strict=True)\n        print(f"continúa desde {os.path.basename(a.desde)} (época {ck0.get(\'epoca\')}, IoU val {ck0.get(\'iou_val\')})")\n    ema = None\n    if a.ema > 0:\n        import copy\n        ema = copy.deepcopy(modelo).eval()\n        for q in ema.parameters():\n            q.requires_grad_(False)\n    opt = torch.optim.AdamW(modelo.parameters(), lr=a.lr, weight_decay=a.wd)\n    if a.calentamiento > 0 or a.lr_min > 0:\n        rel = a.lr_min / a.lr\n        sched = torch.optim.lr_scheduler.LambdaLR(\n            opt, lambda e: factor_lr(e, a.epocas, a.calentamiento, rel))\n    else:\n        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=a.epocas)\n    escalador = torch.amp.GradScaler(\'cuda\', enabled=amp)\n    ruta_ckpt = os.path.join(a.salida, f\'mejor_{base}.pt\')\n    carpeta_top = os.path.join(a.salida, f\'mejores_epocas_{base}\')\n    top = []                                   # [(iou_val, época, ruta)] de las K mejores\n    historial = []\n    mejor, mejor_ep, sin_mejora = -1.0, 0, 0\n    ep_inicial = 1\n    if a.reanudar and os.path.exists(ruta_ultimo):\n        u = torch.load(ruta_ultimo, map_location=disp, weights_only=False)\n        if u.get(\'firma\') != firma:\n            raise SystemExit(f\'{os.path.basename(ruta_ultimo)} es de otra configuración\')\n        modelo.load_state_dict(u[\'modelo\'])\n        if ema is not None and u.get(\'ema\') is not None:\n            ema.load_state_dict(u[\'ema\'])\n        opt.load_state_dict(u[\'opt\']); sched.load_state_dict(u[\'sched\']); escalador.load_state_dict(u[\'escalador\'])\n        rng.bit_generator.state = u[\'rng\']\n        torch.set_rng_state(u[\'torch_rng\'].cpu())\n        if disp == \'cuda\' and u.get(\'cuda_rng\') is not None:\n            torch.cuda.set_rng_state(u[\'cuda_rng\'].cpu())\n        historial, top = u[\'historial\'], [tuple(t) for t in u[\'top\']]\n        mejor, mejor_ep, sin_mejora = u[\'mejor\'], u[\'mejor_ep\'], u[\'sin_mejora\']\n        ep_inicial = u[\'epoca\'] + 1\n        print(f\'retoma desde la época {u["epoca"]} (mejor hasta ahora: época {mejor_ep}, IoU val {mejor:.4f})\')\n\n    def guardar_avance(ep):\n        guardar_seguro({\'firma\': firma, \'epoca\': ep, \'modelo\': modelo.state_dict(),\n                        \'ema\': ema.state_dict() if ema is not None else None,\n                        \'opt\': opt.state_dict(), \'sched\': sched.state_dict(),\n                        \'escalador\': escalador.state_dict(), \'rng\': rng.bit_generator.state,\n                        \'torch_rng\': torch.get_rng_state(),\n                        \'cuda_rng\': torch.cuda.get_rng_state() if disp == \'cuda\' else None,\n                        \'historial\': historial, \'top\': top, \'mejor\': mejor, \'mejor_ep\': mejor_ep,\n                        \'sin_mejora\': sin_mejora}, ruta_ultimo)\n\n    for ep in range(ep_inicial, a.epocas + 1):\n        modelo.train(); perdidas = []\n        orden = rng.choice(len(Xtr), len(Xtr), p=p_muestra) if p_muestra is not None else rng.permutation(len(Xtr))\n        te0 = time.time()\n        for i in range(0, len(orden) - 1, a.lote):\n            j = orden[i:i + a.lote]\n            if len(j) < 2:\n                continue                         # BatchNorm necesita al menos 2\n            x, y, v = Xtr[j].copy(), Ytr[j].copy(), Vtr[j]\n            w = Wtr[j].copy() if Wtr is not None else np.ones_like(y)\n            if pegador is not None or a.brillo > 0:\n                for b in range(len(j)):\n                    if pegador is not None and rng.random() < a.copiar_pegar:\n                        x[b], y[b], w[b] = pegador.aplicar(x[b], y[b], w[b])\n                    if a.brillo > 0:\n                        for kk in esp:\n                            f = 1.0 + rng.uniform(-a.brillo, a.brillo)\n                            mu = receta[\'normalizacion\'][\'por_canal\'][canales[kk]][\'media\']\n                            sd = receta[\'normalizacion\'][\'por_canal\'][canales[kk]][\'desv\']\n                            x[b, kk] = x[b, kk] * f + mu * (f - 1.0) / sd\n            k = int(rng.integers(0, 4)); espejo = bool(rng.integers(0, 2))\n            x, y, v, w = (np.rot90(m, k, axes=(-2, -1)) for m in (x, y, v, w))\n            if espejo:\n                x, y, v, w = x[..., ::-1], y[..., ::-1], v[..., ::-1], w[..., ::-1]\n            x, y, v, w = (torch.from_numpy(np.ascontiguousarray(m)).to(disp) for m in (x, y, v, w))\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=\'cuda\', enabled=amp):\n                logits = modelo(x)[:, 0]\n            loss = perdida_mixta(logits.float(), y, v * w if a.peso_tamano > 1 else v, a.peso_dice, a.ohem)\n            if lovasz is not None:\n                loss = loss + a.peso_lovasz * lovasz(logits.float(), torch.where(v > 0, y, torch.full_like(y, 2.0)))\n            escalador.scale(loss).backward()\n            if a.recorte_grad > 0:\n                escalador.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(modelo.parameters(), a.recorte_grad)\n            escalador.step(opt); escalador.update()\n            if ema is not None:\n                with torch.no_grad():\n                    for qe, q in zip(ema.parameters(), modelo.parameters()):\n                        qe.mul_(a.ema).add_(q.detach(), alpha=1.0 - a.ema)\n                    for be, bq in zip(ema.buffers(), modelo.buffers()):\n                        be.copy_(bq)\n            perdidas.append(float(loss.detach()))\n        sched.step()\n        evaluado = ema if ema is not None else modelo\n        if a.seleccion == \'iou_max\':\n            cv_ep = evaluar(evaluado, Xva, Yva, Vva, np.round(np.arange(0.05, 0.96, 0.05), 2), disp)\n            m = max((cv_ep.metricas(i) for i in range(len(cv_ep.u))), key=lambda r: r[\'iou\'])\n        else:\n            m = evaluar(evaluado, Xva, Yva, Vva, [0.5], disp).metricas(0)\n        historial.append({\'epoca\': ep, \'perdida\': round(float(np.mean(perdidas)), 5),\n                          \'lr\': opt.param_groups[0][\'lr\'], **{k: m[k] for k in (\'iou\', \'f1\', \'precision\', \'recall\')},\n                          \'segundos\': round(time.time() - te0, 1)})\n        marca = \'\'\n        if m[\'iou\'] > mejor:\n            mejor, mejor_ep, sin_mejora = m[\'iou\'], ep, 0\n            torch.save({\'modelo\': evaluado.state_dict(), \'arquitectura\': a.arquitectura,\n                        \'epoca\': ep, \'iou_val\': m[\'iou\'], \'firma\': firma, \'config\': config},\n                       ruta_ckpt)\n            marca = \'  <- mejor, guardado\'\n        else:\n            sin_mejora += 1\n        if a.promediar > 0 and (len(top) < a.promediar or m[\'iou\'] > min(t[0] for t in top)):\n            os.makedirs(carpeta_top, exist_ok=True)\n            ruta_ep = os.path.join(carpeta_top, f\'epoca_{ep:04d}.pt\')\n            guardar_seguro({k: v.detach().cpu().clone() for k, v in evaluado.state_dict().items()}, ruta_ep)\n            top.append((m[\'iou\'], ep, ruta_ep))\n            top.sort(key=lambda t: -t[0])\n            for _, _, r in top[a.promediar:]:\n                if os.path.exists(r):\n                    os.remove(r)\n            top = top[:a.promediar]\n        chicos = \'\'\n        if a.peso_tamano > 1 or a.copiar_pegar > 0 or a.sobremuestreo > 1:\n            evaluado.eval()\n            with torch.no_grad():\n                pv = np.concatenate([torch.sigmoid(evaluado(torch.from_numpy(Xva[i:i + 32]).to(disp))[:, 0]).float().cpu().numpy()\n                                     for i in range(0, len(Xva), 32)])\n            r = recall_por_tamano(pv, Yva, Vva, m[\'umbral\'], objs_va)\n            chicos = f" | pequeños <0.1 ha {r[\'<10\']*100:.0f} %, 0.1-0.5 ha {r[\'10-50\']*100:.0f} %"\n        print(f"época {ep:3d} | pérdida {np.mean(perdidas):.4f} | IoU val {m[\'iou\']:.4f} | "\n              f"F1 {m[\'f1\']:.4f}{chicos} | lr {opt.param_groups[0][\'lr\']:.1e} | {time.time()-te0:.0f} s{marca}",\n              flush=True)\n        if a.reanudar and (ep % max(a.guardar_cada, 1) == 0 or ep == a.epocas):\n            guardar_avance(ep)\n        if sin_mejora >= a.paciencia:\n            print(f\'sin mejora en {a.paciencia} épocas: se detiene\'); break\n    with open(os.path.join(a.salida, f\'historial_{base}.csv\'), \'w\', newline=\'\') as f:\n        w = csv.DictWriter(f, fieldnames=list(historial[0])); w.writeheader(); w.writerows(historial)\n\n    # el mejor modelo, el umbral elegido en validación y una sola medición en prueba\n    umbrales = np.round(np.arange(0.05, 0.96, 0.05), 2)\n    ck = torch.load(ruta_ckpt, map_location=disp, weights_only=False)\n    modelo.load_state_dict(ck[\'modelo\'])\n    cv = evaluar(modelo, Xva, Yva, Vva, umbrales, disp)\n    barrido = [cv.metricas(i) for i in range(len(umbrales))]\n    elegido = max(barrido, key=lambda r: r[\'iou\'])\n    promedio = None\n    if a.promediar > 1 and len(top) > 1:\n        # promedio de pesos de las K mejores épocas, con BatchNorm medido de nuevo;\n        # se queda solo si en validación supera a la mejor época sola\n        estados = [torch.load(r, map_location=\'cpu\', weights_only=False) for _, _, r in top]\n        modelo.load_state_dict(promedio_de_pesos(estados))\n        modelo.to(disp)\n        recalcular_bn(modelo, Xtr, disp)\n        cvp = evaluar(modelo, Xva, Yva, Vva, umbrales, disp)\n        barrido_p = [cvp.metricas(i) for i in range(len(umbrales))]\n        elegido_p = max(barrido_p, key=lambda r: r[\'iou\'])\n        eps = sorted(t[1] for t in top)\n        print(f"\\npromedio de las épocas {eps}: IoU val {elegido_p[\'iou\']:.4f} "\n              f"(la mejor época sola: {elegido[\'iou\']:.4f})")\n        promedio = {\'epocas\': eps, \'iou_val\': elegido_p[\'iou\'], \'iou_val_mejor_epoca\': elegido[\'iou\'],\n                    \'elegido\': elegido_p[\'iou\'] > elegido[\'iou\']}\n        if promedio[\'elegido\']:\n            print(\'  -> se usa el promedio\')\n            barrido, elegido = barrido_p, elegido_p\n            guardar_seguro({\'modelo\': modelo.state_dict(), \'arquitectura\': a.arquitectura,\n                            \'epoca\': mejor_ep, \'promedio_de_epocas\': eps, \'iou_val\': elegido_p[\'iou\'],\n                            \'firma\': firma, \'config\': config}, ruta_ckpt)\n        else:\n            print(\'  -> se usa la mejor época sola\')\n            modelo.load_state_dict(ck[\'modelo\'])\n    prueba = evaluar(modelo, Xte, Yte, Vte, [elegido[\'umbral\']], disp).metricas(0)\n    pt = predecir(modelo, Xte, disp)\n    por_tamano = recall_por_tamano(pt, Yte, Vte, elegido[\'umbral\'])\n    # con TTA (imagen y sus tres espejos): umbral propio, elegido también en validación\n    cvt = evaluar(modelo, Xva, Yva, Vva, umbrales, disp, tta=4)\n    elegido_t = max((cvt.metricas(i) for i in range(len(umbrales))), key=lambda r: r[\'iou\'])\n    prueba_t = evaluar(modelo, Xte, Yte, Vte, [elegido_t[\'umbral\']], disp, tta=4).metricas(0)\n    por_tamano_t = recall_por_tamano(predecir(modelo, Xte, disp, tta=4), Yte, Vte, elegido_t[\'umbral\'])\n    resultados = {\'firma\': firma, \'arquitectura\': a.arquitectura, \'mejor_epoca\': mejor_ep,\n                  \'umbral_operacion\': elegido[\'umbral\'], \'validacion\': elegido,\n                  \'prueba\': prueba, \'prueba_deslizamientos_detectados_por_tamano_px\': por_tamano,\n                  \'barrido_validacion\': barrido, \'config\': config,\n                  \'epocas_corridas\': len(historial),\n                  **({\'promedio_de_pesos\': promedio} if promedio else {}),\n                  \'tta4\': {\'umbral_operacion\': elegido_t[\'umbral\'], \'validacion\': elegido_t,\n                           \'prueba\': prueba_t, \'prueba_deslizamientos_detectados_por_tamano_px\': por_tamano_t},\n                  \'minutos\': round((time.time() - t0) / 60, 1)}\n    json.dump(resultados, open(ruta_res, \'w\', encoding=\'utf-8\'), indent=1, ensure_ascii=False)\n    print(f"\\nmejor época {mejor_ep}; umbral {elegido[\'umbral\']} (elegido en validación)")\n    print(f"validación: IoU {elegido[\'iou\']:.4f}  F1 {elegido[\'f1\']:.4f}")\n    print(f"prueba    : IoU {prueba[\'iou\']:.4f}  F1 {prueba[\'f1\']:.4f}  "\n          f"precisión {prueba[\'precision\']:.4f}  exhaustividad {prueba[\'recall\']:.4f}")\n    print(\'deslizamientos detectados en prueba por tamaño (px a 10 m):\', por_tamano)\n    print(f"con TTA (umbral {elegido_t[\'umbral\']}): IoU val {elegido_t[\'iou\']:.4f} | prueba IoU {prueba_t[\'iou\']:.4f}  "\n          f"F1 {prueba_t[\'f1\']:.4f}")\n    if os.path.exists(ruta_ultimo):\n        os.remove(ruta_ultimo)                 # terminado: el avance ya no hace falta\n    if os.path.isdir(carpeta_top):\n        import shutil\n        shutil.rmtree(carpeta_top)             # las K épocas ya se usaron; ocupan mucho en Drive\n    print(\'archivos:\', ruta_ckpt, ruta_stats, ruta_res, sep=\'\\n  \')\n\n\nif __name__ == \'__main__\':\n    main()\n'
ARCHIVOS['herramientas/ajuste_fino.py'] = '"""\nAjuste fino de un modelo ya entrenado (U-Net v4) con los parches marcados de La Convención.\n\nParte del punto de control de entrenar.py, sigue entrenando con tasa baja mezclando en cada\nlote parches locales y parches de Landslide4Sense (para que no olvide lo que ya sabe) y mide\nel antes y el después sobre las MISMAS zonas de validación local, que el modelo no ve al\nentrenar, y sobre la prueba oficial de Landslide4Sense.\n\nQué hace, en orden:\n\n  1. Lee el punto de control base y sus estadísticas de normalización. NO se recalculan:\n     el modelo ajustado usa la misma receta de entrada, así el servidor no cambia.\n  2. Arma los parches locales con `recetas.desde_l2a` y escala_laconvencion.json, la MISMA\n     función y la misma escala que usa el servidor al detectar.\n  3. Usa solo parches con máscara de nivel 1 (marcado humano). En la máscara, 255 se ignora.\n     La partición sale de la columna «conjunto» de indice_parches.csv (por zona).\n     Con --debiles se suman, solo para entrenar y con peso bajo, parches con máscara de\n     nivel 2 (propuestas de INGEMMET) que tengan alguna pieza aceptada.\n  4. Mide el modelo base (época 0) y después cada época, en validación local y en la\n     validación de Landslide4Sense.\n     BatchNorm queda fijo (medias y varianzas del modelo base) salvo --bn-libre.\n  5. Elige la época con mejor IoU en validación local entre las que no pierden más de\n     --tolerancia-l4s de IoU en la validación de Landslide4Sense. La época 0 compite: si\n     nada la supera, se queda el modelo base y no se escribe uno nuevo.\n  6. Elige el umbral en validación (local si hay cicatrices suficientes; si no, el de L4S)\n     y mide una sola vez la prueba oficial de Landslide4Sense con el base y con el ajustado.\n  7. Da el intervalo de la mejora local remuestreando ZONAS (no píxeles ni parches).\n\nOjo al leer el resultado: la validación local se usa para elegir época y umbral, así que su\nIoU es optimista. Con pocas cicatrices el intervalo es ancho y el resultado es exploratorio;\nel script lo dice. Para una cifra final hace falta un tercer grupo de zonas que no se use\npara decidir nada.\n\nUso (Colab con GPU):\n  python3 ajuste_fino.py --l4s /content/l4s \\\n      --desde  .../mejor_unet_b1b4b13b14_b2809be440d6.pt \\\n      --stats  .../stats_norm_b2809be440d6.json \\\n      --parches /content/local/parches_128 --mascaras /content/local/mascaras_nivel1/parches_128 \\\n      --salida /content/drive/MyDrive/ajuste_fino\n\nDeja, con la misma forma que entrenar.py (exportar_onnx.py los lee sin cambios):\n  mejor_<arq>_b1b4b13b14_<firma>.pt   stats_norm_<firma>.json   resultados_<arq>_b1b4b13b14_<firma>.json\n  historial_ajuste_<firma>.csv\n"""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport sys\nimport time\n\nimport numpy as np\n\nAQUI = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, AQUI)\nsys.path.insert(0, os.path.join(AQUI, \'..\', \'backend\'))\nsys.path.insert(0, os.path.join(AQUI, \'..\'))\n\nfrom entrenar import (ETIQUETA_CANALES, Conteo, cargar, factor_lr, modelo_con_indices, objetos,  # noqa: E402\n                      pares, perdida_mixta, predecir, receta_de)\n\nUMBRALES = np.round(np.arange(0.05, 0.96, 0.05), 2)\nBANDAS = [\'B01\', \'B02\', \'B03\', \'B04\']\n\n\n# ------------------------------------------------------------------- datos locales\ndef cargar_locales(carpeta_parches, carpeta_mascaras, receta, escala, conjunto_permitido, solo_con_positivos=False):\n    """Parches locales con máscara. Devuelve X, Y, V y la lista de filas del índice.\n    V = 1 donde el píxel cuenta: con dato y máscara distinta de 255."""\n    import rasterio\n    from nucleo import recetas\n    filas = []\n    indice_amp = os.path.join(carpeta_parches, \'indice_parches_ampliado.csv\')\n    indice = indice_amp if os.path.exists(indice_amp) else os.path.join(carpeta_parches, \'indice_parches.csv\')\n    if not os.path.exists(indice):\n        raise SystemExit(f\'falta {indice}: corre cortar_parches_128.py\')\n    with open(indice, encoding=\'utf-8\') as f_ind:\n        filas_ind = list(csv.DictReader(f_ind))\n\n    # Si existe catalogo_parches_tiers.csv, leer pesos estratificados por parche (Oro=1.0, HardNeg=0.8, Plata=0.5, Bronce=0.3/0.2)\n    pesos_tiers = None\n    posibles_tiers = [\n        os.path.join(os.path.dirname(carpeta_mascaras.rstrip(\'/\\\\\')), \'catalogo_parches_tiers.csv\'),\n        os.path.join(carpeta_mascaras, \'..\', \'catalogo_parches_tiers.csv\'),\n    ]\n    for rt in posibles_tiers:\n        if os.path.exists(rt):\n            with open(rt, encoding=\'utf-8\') as ft:\n                pesos_tiers = {\n                    (row[\'proyecto\'], row[\'parche\']): float(row[\'peso_perdida\'])\n                    for row in csv.DictReader(ft)\n                    if float(row.get(\'peso_perdida\') or 0) > 0\n                }\n            break\n\n    # Si existe mascaras_nivel2_completo.csv en el padre de carpeta_mascaras, respetar incluir_debil\n    permitidos_n2 = None\n    posibles_csv = [\n        os.path.join(os.path.dirname(carpeta_mascaras.rstrip(\'/\\\\\')), \'mascaras_nivel2_completo.csv\'),\n        os.path.join(carpeta_mascaras, \'..\', \'mascaras_nivel2_completo.csv\'),\n    ]\n    for rc in posibles_csv:\n        if os.path.exists(rc):\n            with open(rc, encoding=\'utf-8\') as fc:\n                permitidos_n2 = {\n                    (row[\'proyecto\'], row[\'tesela\'])\n                    for row in csv.DictReader(fc)\n                    if row.get(\'incluir_debil\') == \'si\'\n                }\n            break\n\n    for r in filas_ind:\n        if r[\'conjunto\'] not in conjunto_permitido:\n            continue\n        m = os.path.join(carpeta_mascaras, r[\'proyecto\'], r[\'parche\'] + \'.tif\')\n        es_raiz_n2 = False\n        if not os.path.exists(m):\n            m = os.path.join(carpeta_mascaras, r[\'parche\'] + \'.tif\')          # nivel 2 clásico: sin subcarpeta\n            es_raiz_n2 = True\n            if not os.path.exists(m) or r[\'proyecto\'] != \'datos_ingemmet\':\n                continue\n        peso_p = None\n        if solo_con_positivos and not es_raiz_n2:\n            if pesos_tiers is not None:\n                if (r[\'proyecto\'], r[\'parche\']) not in pesos_tiers:\n                    continue\n                peso_p = pesos_tiers[(r[\'proyecto\'], r[\'parche\'])]\n            elif permitidos_n2 is not None:\n                if (r[\'proyecto\'], r[\'tesela\']) not in permitidos_n2:\n                    continue\n        filas.append(dict(r, _mascara=m, _imagen=os.path.join(carpeta_parches, r[\'archivo\']),\n                          _es_raiz_n2=es_raiz_n2, _peso_tier=peso_p))\n    X, Y, V, usadas = [], [], [], []\n    for r in filas:\n        with rasterio.open(r[\'_mascara\']) as d:\n            mk = d.read(1)\n        if solo_con_positivos and (pesos_tiers is None and permitidos_n2 is None or r.get(\'_es_raiz_n2\')) and not (mk == 1).any():\n            continue\n        with rasterio.open(r[\'_imagen\']) as d:\n            x = d.read().astype(np.float32)\n        if x.shape != (6, 128, 128) or mk.shape != (128, 128):\n            raise SystemExit(f"{r[\'parche\']}: tamaño inesperado {x.shape} / {mk.shape}")\n        valido = (x[1] > 0) & (mk != 255)\n        t = recetas.desde_l2a({b: x[k] for k, b in enumerate(BANDAS)},\n                              {\'pendiente\': x[4], \'elevacion\': x[5]}, valido, receta, escala=escala)\n        X.append(t); Y.append((mk == 1).astype(np.float32)); V.append(valido.astype(np.float32)); usadas.append(r)\n    if not X:\n        z = np.zeros((0, len(receta[\'canales\']), 128, 128), np.float32)\n        return z, np.zeros((0, 128, 128), np.float32), np.zeros((0, 128, 128), np.float32), []\n    return np.stack(X), np.stack(Y), np.stack(V), usadas\n\n\n# ------------------------------------------------------------------- medición\ndef barrido(prob, Y, V):\n    c = Conteo(UMBRALES)\n    for i in range(0, len(prob), 64):\n        c.sumar(prob[i:i + 64], Y[i:i + 64], V[i:i + 64])\n    return [c.metricas(i) for i in range(len(UMBRALES))]\n\n\ndef en_umbral(lista, u):\n    return min(lista, key=lambda r: abs(r[\'umbral\'] - u))\n\n\ndef resumen_local(prob, Y, V, filas, u):\n    """Métricas locales a un umbral: píxeles, cicatrices tocadas y falsas alarmas en negativos."""\n    m = en_umbral(barrido(prob, Y, V), u)\n    objs = objetos(Y * V)\n    tocadas = sum(bool(((prob[i][sl] >= u) & (V[i][sl] > 0) & mm).any()) for i, sl, mm, _ in objs)\n    teselas = {}\n    for i, r in enumerate(filas):\n        t = teselas.setdefault((r[\'proyecto\'], r[\'tesela\']), [0.0, 0.0, 0.0])\n        t[0] += float((Y[i] * V[i]).sum()); t[1] += float(((prob[i] >= u) & (V[i] > 0)).sum()); t[2] += float(V[i].sum())\n    neg = [t for t in teselas.values() if t[0] == 0]\n    px_neg, fa_neg = sum(t[2] for t in neg), sum(t[1] for t in neg)\n    return dict(m, cicatrices=len(objs), cicatrices_tocadas=tocadas, teselas=len(teselas),\n                teselas_negativas=len(neg),\n                falsas_alarmas_ha_por_km2_en_negativos=round(fa_neg / max(px_neg, 1) * 100, 3))\n\n\ndef regla_suelo_pendiente(X, receta, pend_min=10.0, b4_min_l4s=1.29):\n    """Comparador sin aprendizaje (regla espectral y de pendiente, análoga a Devara 2024):\n    marca 1,0 donde hay suelo desnudo (GRVI <= 0 y B4 >= 1,29 en unidades L4S, ~600 DN)\n    sobre ladera (pendiente >= pend_min grados)."""\n    if len(X) == 0:\n        return np.zeros((0, 128, 128), np.float32)\n    can = receta[\'canales\']\n    pc = receta[\'normalizacion\'][\'por_canal\']\n    crudo = lambda c: X[:, can.index(c)] * float(pc[c][\'desv\']) + float(pc[c][\'media\'])\n    g, r, p = crudo(\'B03\'), crudo(\'B04\'), crudo(\'pendiente\') * 20.292\n    grvi = (g - r) / (np.abs(g + r) + 1e-3)\n    return ((grvi <= 0.0) & (r >= b4_min_l4s) & (p >= pend_min)).astype(np.float32)\n\n\ndef bootstrap_zonas(prob_a, u_a, prob_b, u_b, Y, V, filas, n=2000, semilla=7):\n    """Intervalo del 95 % de IoU(b) − IoU(a) remuestreando zonas completas."""\n    zonas = sorted({r[\'zona\'] for r in filas})\n    if len(zonas) < 2:\n        return None\n    cuenta = {z: np.zeros((2, 3)) for z in zonas}\n    for i, r in enumerate(filas):\n        ok, t = V[i] > 0, Y[i] > 0\n        for k, (p, u) in enumerate(((prob_a, u_a), (prob_b, u_b))):\n            mm = p[i] >= u\n            cuenta[r[\'zona\']][k] += [np.sum(mm & t & ok), np.sum(mm & ~t & ok), np.sum(~mm & t & ok)]\n    C = np.stack([cuenta[z] for z in zonas])\n    rng = np.random.default_rng(semilla)\n    difs = []\n    for _ in range(n):\n        s = C[rng.integers(0, len(zonas), len(zonas))].sum(0)\n        iou = s[:, 0] / np.maximum(s.sum(1), 1)\n        difs.append(iou[1] - iou[0])\n    lo, hi = np.percentile(difs, [2.5, 97.5])\n    return {\'zonas\': len(zonas), \'ic95\': [round(float(lo), 4), round(float(hi), 4)],\n            \'gana_pct\': round(float(np.mean(np.array(difs) > 0)) * 100, 1)}\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__.split(\'\\n\\n\')[0])\n    ap.add_argument(\'--l4s\', required=True)\n    ap.add_argument(\'--desde\', required=True, help=\'punto de control .pt del modelo base (entrenar.py)\')\n    ap.add_argument(\'--stats\', required=True, help=\'stats_norm_<firma>.json del modelo base\')\n    ap.add_argument(\'--parches\', required=True, help=\'carpeta parches_128 con indice_parches.csv\')\n    ap.add_argument(\'--mascaras\', required=True, help=\'mascaras_nivel1/parches_128\')\n    ap.add_argument(\'--debiles\', help=\'(opcional) mascaras_provisionales/nivel2/parches_128, solo para entrenar\')\n    ap.add_argument(\'--peso-debil\', dest=\'peso_debil\', type=float, default=0.3)\n    ap.add_argument(\'--escala\', help=\'escala_laconvencion.json; por defecto la del servidor (nucleo.config)\')\n    ap.add_argument(\'--salida\', required=True)\n    ap.add_argument(\'--epocas\', type=int, default=40)\n    ap.add_argument(\'--lote\', type=int, default=16)\n    ap.add_argument(\'--lr\', type=float, default=3e-5, help=\'diez veces menor que en el entrenamiento original\')\n    ap.add_argument(\'--weight-decay\', dest=\'wd\', type=float, default=1e-4)\n    ap.add_argument(\'--frac-local\', dest=\'frac_local\', type=float, default=0.5, help=\'parte de cada lote que es local\')\n    ap.add_argument(\'--pasos\', type=int, default=0, help=\'lotes por época (0 = según la cantidad de parches locales)\')\n    ap.add_argument(\'--ohem\', type=float, default=None, help=\'por defecto, el del modelo base\')\n    ap.add_argument(\'--congelar-encoder\', dest=\'congelar\', action=\'store_true\',\n                    help=\'entrena solo el decodificador (más seguro con muy pocos datos)\')\n    ap.add_argument(\'--bn-libre\', dest=\'bn_libre\', action=\'store_true\',\n                    help=\'deja que BatchNorm recalcule sus medias (solo con cientos de parches locales)\')\n    ap.add_argument(\'--tolerancia-l4s\', dest=\'tol\', type=float, default=0.03,\n                    help=\'IoU que se acepta perder en la validación de Landslide4Sense\')\n    ap.add_argument(\'--min-cicatrices\', dest=\'min_cic\', type=int, default=10,\n                    help=\'cicatrices mínimas en validación local para elegir ahí el umbral\')\n    ap.add_argument(\'--umbral-base\', dest=\'umbral_base\', type=float, default=0.45,\n                    help=\'umbral de la receta del modelo base, para comparar a umbral fijo\')\n    ap.add_argument(\'--geomorfologia\', action=\'store_true\',\n                    help=\'añade CURVATURA y TPI calculados desde el DEM como canales extra (Vía 3)\')\n    ap.add_argument(\'--semilla\', type=int, default=42)\n    ap.add_argument(\'--limite\', type=int, help=\'solo para probar: parches de L4S por conjunto\')\n    a = ap.parse_args()\n\n    import torch\n    import segmentation_models_pytorch as smp\n    from nucleo import config as cfg_srv\n\n    torch.manual_seed(a.semilla); np.random.seed(a.semilla)\n    rng = np.random.default_rng(a.semilla)\n    disp = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    amp = disp == \'cuda\'\n    os.makedirs(a.salida, exist_ok=True)\n    t0 = time.time()\n\n    ck0 = torch.load(a.desde, map_location=\'cpu\', weights_only=False)\n    cfg0 = ck0.get(\'config\') or {}\n    arq = ck0.get(\'arquitectura\', \'unet\')\n    canales_base = list(cfg0.get(\'canales\') or [\'B01\', \'B02\', \'B03\', \'B04\', \'pendiente\', \'elevacion\'])\n    if canales_base != [\'B01\', \'B02\', \'B03\', \'B04\', \'pendiente\', \'elevacion\']:\n        raise SystemExit(f\'el modelo base usa los canales {canales_base}; los parches locales traen B1 a B4, pendiente y DEM\')\n    canales = list(canales_base)\n    stats = json.load(open(a.stats, encoding=\'utf-8\'))\n    if a.geomorfologia:\n        canales.extend([\'CURVATURA\', \'TPI\'])\n        stats[\'CURVATURA\'] = {\'media\': 0.0, \'desv\': 1.0}\n        stats[\'TPI\'] = {\'media\': 0.0, \'desv\': 1.0}\n    receta = receta_de(canales, stats)\n    escala = json.load(open(a.escala, encoding=\'utf-8\'))[\'media_dn\'] if a.escala else cfg_srv.ESCALA_L4S\n    if not escala:\n        raise SystemExit(\'no hay escala de bandas: pasa --escala escala_laconvencion.json\')\n    ohem = a.ohem if a.ohem is not None else float((cfg0.get(\'v4\') or {}).get(\'ohem\', 0.0))\n    indices = bool((cfg0.get(\'v4\') or {}).get(\'indices_verdor\')) or any(k.startswith(\'base.\') for k in ck0[\'modelo\'])\n    print(f"base: {os.path.basename(a.desde)} | {arq} | época {ck0.get(\'epoca\')} | IoU val {ck0.get(\'iou_val\')} | "\n          f"índices de verdor {\'sí\' if indices else \'no\'} | geomorfología {\'sí\' if a.geomorfologia else \'no\'} | ohem {ohem} | {disp}")\n\n    # ---------------------------------------------------------------- datos\n    tr, va, te = (pares(a.l4s, s, a.limite) for s in (\'train\', \'validation\', \'test\'))\n    Xtr, Ytr, Vtr = cargar(tr, receta)\n    Xva, Yva, Vva = cargar(va, receta)\n    Xte, Yte, Vte = cargar(te, receta)\n    Xl, Yl, Vl, Fl = cargar_locales(a.parches, a.mascaras, receta, escala, {\'entrenamiento\'})\n    Xv, Yv, Vv, Fv = cargar_locales(a.parches, a.mascaras, receta, escala, {\'validacion\'})\n    Wl = np.ones(len(Xl), np.float32)\n    n_debiles = 0\n    if a.debiles:\n        con_n1 = {r[\'parche\'] for r in Fl}\n        Xd, Yd, Vd, Fd = cargar_locales(a.parches, a.debiles, receta, escala, {\'entrenamiento\'}, solo_con_positivos=True)\n        quedan = [i for i, r in enumerate(Fd) if r[\'parche\'] not in con_n1]\n        if quedan:\n            Xl = np.concatenate([Xl, Xd[quedan]]); Yl = np.concatenate([Yl, Yd[quedan]]); Vl = np.concatenate([Vl, Vd[quedan]])\n            pesos_q = np.array([Fd[i].get(\'_peso_tier\') or a.peso_debil for i in quedan], dtype=np.float32)\n            Wl = np.concatenate([Wl, pesos_q])\n            Fl = Fl + [Fd[i] for i in quedan]\n            n_debiles = len(quedan)\n    zonas_tr, zonas_va = {r[\'zona\'] for r in Fl}, {r[\'zona\'] for r in Fv}\n    if zonas_tr & zonas_va:\n        raise SystemExit(f\'zonas en entrenamiento y validación a la vez: {sorted(zonas_tr & zonas_va)}\')\n    if len(Xl) < 4:\n        raise SystemExit(f\'solo hay {len(Xl)} parches locales de entrenamiento con máscara: faltan teselas marcadas\')\n    cic_tr, cic_va = len(objetos(Yl * Vl)), len(objetos(Yv * Vv))\n    print(f\'L4S: entrenamiento {len(Xtr)}, validación {len(Xva)}, prueba {len(Xte)}\')\n    print(f\'local entrenamiento: {len(Xl)} parches ({n_debiles} débiles con peso {a.peso_debil}), {len(zonas_tr)} zonas, \'\n          f\'{cic_tr} cicatrices, {float((Yl * Vl).sum()) * 0.01:.1f} ha\')\n    print(f\'local validación  : {len(Xv)} parches, {len(zonas_va)} zonas, {cic_va} cicatrices, \'\n          f\'{float((Yv * Vv).sum()) * 0.01:.1f} ha\')\n    hay_val = len(Xv) > 0\n    hay_pos_val = cic_va > 0\n    if not hay_val:\n        print(\'AVISO: no hay validación local marcada. Se entrena, pero la época se elige solo con Landslide4Sense \'\n              \'y NO se puede afirmar ninguna mejora en La Convención.\')\n    elif cic_va < a.min_cic:\n        print(f\'AVISO: la validación local tiene {cic_va} cicatrices (menos de {a.min_cic}): resultado exploratorio.\')\n\n    # ---------------------------------------------------------------- modelo\n    clase = {\'unet\': smp.Unet, \'deeplabv3plus\': smp.DeepLabV3Plus}[arq]\n    modelo = clase(encoder_name=cfg0.get(\'encoder\', \'resnet34\'), encoder_weights=None,\n                   in_channels=len(canales) + (2 if indices else 0), classes=1)\n    if indices:\n        modelo = modelo_con_indices(modelo, canales)\n    sd = dict(ck0[\'modelo\'])\n    if a.geomorfologia:\n        clave_conv = \'base.encoder.conv1.weight\' if indices else \'encoder.conv1.weight\'\n        if clave_conv in sd:\n            w_old = sd[clave_conv]\n            w_new = torch.zeros((w_old.shape[0], w_old.shape[1] + 2, w_old.shape[2], w_old.shape[3]), dtype=w_old.dtype)\n            w_new[:, :6] = w_old[:, :6]\n            if w_old.shape[1] > 6:\n                w_new[:, 8:] = w_old[:, 6:]\n            sd[clave_conv] = w_new\n    modelo.load_state_dict(sd, strict=True)\n    modelo = modelo.to(disp)\n    nucleo_red = modelo.base if indices else modelo\n    if a.congelar:\n        for q in nucleo_red.encoder.parameters():\n            q.requires_grad_(False)\n\n    def medir(m):\n        pv = predecir(m, Xva, disp)\n        b_l4s = barrido(pv, Yva, Vva)\n        out = {\'l4s\': max(b_l4s, key=lambda r: r[\'iou\']), \'l4s_barrido\': b_l4s}\n        if hay_val:\n            pl = predecir(m, Xv, disp)\n            b_loc = barrido(pl, Yv, Vv)\n            out[\'local_barrido\'] = b_loc\n            out[\'local\'] = max(b_loc, key=lambda r: r[\'iou\']) if hay_pos_val else min(b_loc, key=lambda r: (r[\'fp\'], r[\'umbral\']))\n        return out\n\n    def puntaje(m):\n        if not hay_val:\n            return m[\'l4s\'][\'iou\']\n        if hay_pos_val:\n            return m[\'local\'][\'iou\']\n        fp05 = en_umbral(m[\'local_barrido\'], 0.5)\n        return -fp05[\'fp\'] / max(fp05[\'fp\'] + fp05[\'tn\'], 1)\n\n    m0 = medir(modelo)\n    piso_l4s = m0[\'l4s\'][\'iou\'] - a.tol\n    print(f"época  0 (modelo base) | IoU val L4S {m0[\'l4s\'][\'iou\']:.4f} (umbral {m0[\'l4s\'][\'umbral\']})"\n          + (f" | IoU val local {m0[\'local\'][\'iou\']:.4f} (umbral {m0[\'local\'][\'umbral\']})" if hay_val and hay_pos_val else \'\'))\n    config = dict(cfg0, ajuste_fino={\'desde\': os.path.basename(a.desde), \'firma_base\': ck0.get(\'firma\'),\n                                     \'lr\': a.lr, \'lote\': a.lote, \'frac_local\': a.frac_local, \'epocas_max\': a.epocas,\n                                     \'ohem\': ohem, \'congelar_encoder\': a.congelar, \'bn_libre\': a.bn_libre, \'tolerancia_l4s\': a.tol,\n                                     \'semilla\': a.semilla, \'parches_locales\': len(Xl), \'parches_debiles\': n_debiles,\n                                     \'peso_debil\': a.peso_debil if n_debiles else None,\n                                     \'zonas_entrenamiento\': sorted(zonas_tr), \'zonas_validacion\': sorted(zonas_va)})\n    firma = hashlib.sha1(json.dumps(dict(config, arquitectura=arq), sort_keys=True).encode()).hexdigest()[:12]\n    base = f\'{arq}_{ETIQUETA_CANALES}_{firma}\'\n    ruta_ckpt = os.path.join(a.salida, f\'mejor_{base}.pt\')\n\n    params = [q for q in modelo.parameters() if q.requires_grad]\n    opt = torch.optim.AdamW(params, lr=a.lr, weight_decay=a.wd)\n    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda e: factor_lr(e, a.epocas, 2, 0.05))\n    escalador = torch.amp.GradScaler(\'cuda\', enabled=amp)\n    n_loc = max(1, min(a.lote - 1, int(round(a.lote * a.frac_local))))\n    n_l4s = a.lote - n_loc\n    pasos = a.pasos or max(20, math.ceil(len(Xl) / n_loc))\n    historial = [{\'epoca\': 0, \'perdida\': None, \'iou_l4s\': m0[\'l4s\'][\'iou\'],\n                  \'iou_local\': m0[\'local\'][\'iou\'] if hay_val else None, \'puntaje\': puntaje(m0), \'admisible\': True}]\n    mejor = {\'epoca\': 0, \'puntaje\': puntaje(m0), \'medida\': m0, \'estado\': None}\n    cola_l4s = rng.permutation(len(Xtr)); pos = 0\n    for ep in range(1, a.epocas + 1):\n        modelo.train()\n        if not a.bn_libre:\n            # las medias y varianzas de BatchNorm se quedan como en el modelo base: con pocos\n            # parches locales por lote se desajustan y el modelo empeora aunque no aprenda nada\n            for capa in modelo.modules():\n                if isinstance(capa, torch.nn.modules.batchnorm._BatchNorm):\n                    capa.eval()\n        perdidas = []; te0 = time.time()\n        for _ in range(pasos):\n            jl = rng.integers(0, len(Xl), n_loc)\n            if pos + n_l4s > len(cola_l4s):\n                cola_l4s = rng.permutation(len(Xtr)); pos = 0\n            jg = cola_l4s[pos:pos + n_l4s]; pos += n_l4s\n            x = np.concatenate([Xl[jl], Xtr[jg]]); y = np.concatenate([Yl[jl], Ytr[jg]])\n            v = np.concatenate([Vl[jl] * Wl[jl][:, None, None], Vtr[jg]])\n            k = int(rng.integers(0, 4)); espejo = bool(rng.integers(0, 2))\n            x, y, v = (np.rot90(mm, k, axes=(-2, -1)) for mm in (x, y, v))\n            if espejo:\n                x, y, v = x[..., ::-1], y[..., ::-1], v[..., ::-1]\n            x, y, v = (torch.from_numpy(np.ascontiguousarray(mm)).to(disp) for mm in (x, y, v))\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=\'cuda\', enabled=amp):\n                logits = modelo(x)[:, 0]\n            loss = perdida_mixta(logits.float(), y, v, 0.5, ohem)\n            escalador.scale(loss).backward()\n            escalador.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(params, 1.0)\n            escalador.step(opt); escalador.update()\n            perdidas.append(float(loss.detach()))\n        sched.step()\n        m = medir(modelo)\n        admisible = m[\'l4s\'][\'iou\'] >= piso_l4s\n        p = puntaje(m)\n        marca = \'\'\n        if admisible and p > mejor[\'puntaje\'] + 1e-6:\n            mejor = {\'epoca\': ep, \'puntaje\': p, \'medida\': m,\n                     \'estado\': {kk: vv.detach().cpu().clone() for kk, vv in modelo.state_dict().items()}}\n            marca = \'  <- mejor\'\n        historial.append({\'epoca\': ep, \'perdida\': round(float(np.mean(perdidas)), 5), \'iou_l4s\': m[\'l4s\'][\'iou\'],\n                          \'iou_local\': m[\'local\'][\'iou\'] if hay_val else None, \'puntaje\': round(float(p), 5),\n                          \'admisible\': admisible})\n        print(f"época {ep:2d} | pérdida {np.mean(perdidas):.4f} | IoU val L4S {m[\'l4s\'][\'iou\']:.4f}"\n              + (\'\' if admisible else \' (pierde demasiado)\')\n              + (f" | IoU val local {m[\'local\'][\'iou\']:.4f}" if hay_val and hay_pos_val else \'\')\n              + f" | {time.time() - te0:.0f} s{marca}", flush=True)\n    with open(os.path.join(a.salida, f\'historial_ajuste_{firma}.csv\'), \'w\', newline=\'\') as f:\n        w = csv.DictWriter(f, fieldnames=list(historial[0])); w.writeheader(); w.writerows(historial)\n\n    # ---------------------------------------------------------------- medición final\n    base_modelo = clase(encoder_name=cfg0.get(\'encoder\', \'resnet34\'), encoder_weights=None,\n                        in_channels=len(canales) + (2 if indices else 0), classes=1)\n    if indices:\n        base_modelo = modelo_con_indices(base_modelo, canales)\n    base_modelo.load_state_dict(sd, strict=True)\n    base_modelo = base_modelo.to(disp)\n    mejoro = mejor[\'epoca\'] > 0\n    if mejoro:\n        modelo.load_state_dict(mejor[\'estado\'])\n    else:\n        modelo.load_state_dict(sd, strict=True)\n    mF = mejor[\'medida\']\n    usar_local = hay_val and hay_pos_val and cic_va >= a.min_cic\n\n    def umbral_de(m):\n        return m[\'local\'][\'umbral\'] if usar_local else m[\'l4s\'][\'umbral\']\n\n    u0, uF = umbral_de(m0), umbral_de(mF)\n    res = {\'firma\': firma, \'arquitectura\': arq, \'mejor_epoca\': mejor[\'epoca\'], \'mejoro\': mejoro,\n           \'umbral_operacion\': uF, \'umbral_elegido_en\': \'validación local\' if usar_local else \'validación de Landslide4Sense\',\n           \'config\': config, \'epocas_corridas\': a.epocas, \'base\': {\'firma\': ck0.get(\'firma\'), \'umbral\': u0}}\n    # prueba oficial de Landslide4Sense, una vez, con el umbral elegido en SU validación\n    pt0, ptF = predecir(base_modelo, Xte, disp), predecir(modelo, Xte, disp)\n    ptR = regla_suelo_pendiente(Xte, receta)\n    res[\'base\'][\'l4s_validacion\'] = m0[\'l4s\']\n    res[\'base\'][\'l4s_prueba\'] = en_umbral(barrido(pt0, Yte, Vte), m0[\'l4s\'][\'umbral\'])\n    res[\'validacion\'] = mF[\'l4s\']\n    res[\'prueba\'] = en_umbral(barrido(ptF, Yte, Vte), mF[\'l4s\'][\'umbral\'])\n    res[\'comparador_sin_aprendizaje\'] = {\n        \'regla\': \'GRVI <= 0, B4 >= 1.29 L4S (~600 DN) y pendiente >= 10°\',\n        \'l4s_prueba\': en_umbral(barrido(ptR, Yte, Vte), 0.5),\n    }\n    res[\'barrido_validacion\'] = mF[\'l4s_barrido\']\n    if hay_val:\n        pl0, plF = predecir(base_modelo, Xv, disp), predecir(modelo, Xv, disp)\n        plR = regla_suelo_pendiente(Xv, receta)\n        res[\'base\'][\'local_validacion\'] = resumen_local(pl0, Yv, Vv, Fv, u0)\n        res[\'local_validacion\'] = resumen_local(plF, Yv, Vv, Fv, uF)\n        res[\'comparador_sin_aprendizaje\'][\'local_validacion\'] = resumen_local(plR, Yv, Vv, Fv, 0.5)\n        res[\'local_validacion_umbral_receta_base\'] = {\n            \'umbral\': a.umbral_base, \'base\': resumen_local(pl0, Yv, Vv, Fv, a.umbral_base),\n            \'ajustado\': resumen_local(plF, Yv, Vv, Fv, a.umbral_base)}\n        res[\'local_mejora_iou\'] = bootstrap_zonas(pl0, u0, plF, uF, Yv, Vv, Fv) if hay_pos_val else None\n        res[\'local_advertencias\'] = [s for s in (\n            \'la validación local se usó para elegir época y umbral: su IoU es optimista\',\n            f\'solo {cic_va} cicatrices en validación local: resultado exploratorio\' if cic_va < a.min_cic else \'\',\n            f\'solo {len(zonas_va)} zonas de validación\' if len(zonas_va) < 5 else \'\') if s]\n    res[\'minutos\'] = round((time.time() - t0) / 60, 1)\n\n    print(\'\\n\' + \'=\' * 78)\n    if not mejoro:\n        print(\'Ninguna época superó al modelo base sin perder en Landslide4Sense: SE QUEDA EL MODELO BASE.\')\n        print(\'No se escribe un modelo nuevo. Suele querer decir que faltan teselas marcadas.\')\n    else:\n        torch.save({\'modelo\': modelo.state_dict(), \'arquitectura\': arq, \'epoca\': mejor[\'epoca\'],\n                    \'iou_val\': mF[\'l4s\'][\'iou\'], \'firma\': firma, \'config\': config}, ruta_ckpt)\n        json.dump(stats, open(os.path.join(a.salida, f\'stats_norm_{firma}.json\'), \'w\', encoding=\'utf-8\'), indent=1)\n        print(f"mejor época {mejor[\'epoca\']}; umbral {uF} (elegido en {res[\'umbral_elegido_en\']})")\n    f4 = lambda r: f"IoU {r[\'iou\']:.4f}  F1 {r[\'f1\']:.4f}  precisión {r[\'precision\']:.4f}  exhaustividad {r[\'recall\']:.4f}"\n    print(f"Landslide4Sense, prueba oficial   regla   : {f4(res[\'comparador_sin_aprendizaje\'][\'l4s_prueba\'])}")\n    print(f"                                  base    : {f4(res[\'base\'][\'l4s_prueba\'])}")\n    print(f"                                  ajustado: {f4(res[\'prueba\'])}")\n    if hay_val:\n        rg = res[\'comparador_sin_aprendizaje\'][\'local_validacion\']\n        b, j = res[\'base\'][\'local_validacion\'], res[\'local_validacion\']\n        print(f"La Convención, validación local   regla   : {f4(rg)}  | cicatrices {rg[\'cicatrices_tocadas\']}/{rg[\'cicatrices\']} "\n              f"| falsas alarmas {rg[\'falsas_alarmas_ha_por_km2_en_negativos\']} ha/km² en negativos")\n        print(f"                                  base    : {f4(b)}  | cicatrices {b[\'cicatrices_tocadas\']}/{b[\'cicatrices\']} "\n              f"| falsas alarmas {b[\'falsas_alarmas_ha_por_km2_en_negativos\']} ha/km² en negativos")\n        print(f"                                  ajustado: {f4(j)}  | cicatrices {j[\'cicatrices_tocadas\']}/{j[\'cicatrices\']} "\n              f"| falsas alarmas {j[\'falsas_alarmas_ha_por_km2_en_negativos\']} ha/km² en negativos")\n        if mejoro and res.get(\'local_mejora_iou\'):\n            bz = res[\'local_mejora_iou\']\n            print(f"mejora de IoU local, IC 95 % por zonas ({bz[\'zonas\']}): [{bz[\'ic95\'][0]:+.4f}, {bz[\'ic95\'][1]:+.4f}]; "\n                  f"gana en el {bz[\'gana_pct\']} % de los remuestreos")\n        for s in res[\'local_advertencias\']:\n            print(\'  ojo:\', s)\n    json.dump(res, open(os.path.join(a.salida, f\'resultados_{base}.json\'), \'w\', encoding=\'utf-8\'), indent=1, ensure_ascii=False)\n    print(\'archivos en\', a.salida)\n\n\nif __name__ == \'__main__\':\n    main()\n'
ARCHIVOS['herramientas/exportar_onnx.py'] = '"""\nExporta un checkpoint de la Fase 4 a ONNX, con su receta, y comprueba que el\nONNX resultante está entrenado y da lo mismo que PyTorch.\n\nCorrige el fallo de los ONNX del repositorio, que se exportaron sin cargar el\nentrenamiento. Aquí la carga es estricta (cualquier clave que no calce es un\nerror, no un aviso), se comprueba que los pesos cambiaron al cargar, y el ONNX\npasa la misma prueba forense que hace el backend antes de usarlo.\n\nUso, en Colab o en local:\n\n  python3 exportar_onnx.py \\\n      --checkpoint mejor_deeplabv3plus_6ch_23c2a9ca2368.pt \\\n      --stats stats_norm_23c2a9ca2368.json \\\n      --resultados resultados_deeplabv3plus_6ch_23c2a9ca2368.json \\\n      --salida modelos/ --nombre deeplabv3plus \\\n      --autor "Huarcaya Pumacayo, V. N."\n\nPara la U-Net de Leonardo es lo mismo con sus tres archivos y --nombre unet.\nSi el checkpoint no trae la configuración de la Fase 4 (por ejemplo, un\nstate_dict suelto), se dice a mano. Para las entradas de la especificación\n(Sentinel-2 B1 a B4, pendiente B13 y DEM B14 de ALOS PALSAR) y salida de 2 clases:\n\n  python3 exportar_onnx.py --checkpoint unet.pt --salida modelos/ --nombre unet \\\n      --arquitectura unet --encoder resnet34 --lado 128 \\\n      --canales B01,B02,B03,B04,B13,B14 --clases 2 \\\n      --normalizacion media_desv --stats stats_unet.json\n\n--stats acepta {"0": {"lo":..,"hi":..}} (índices de L4S, como la Fase 4),\n{"B01": {"media":..,"desv":..}} o {"mean": [...], "std": [...]} en el orden de\n--canales. Con --normalizacion ninguna no hace falta.\n\nCon --tta 4 el ONNX lleva dentro el promedio de la imagen y sus tres espejos (lo que\nentrenar.py mide como «con TTA») y usa el umbral elegido para eso en validación. El\nservidor no cambia: recibe el mismo tensor y devuelve la misma forma, pero cada parche\ncuesta 4 veces más de cálculo.\nRequiere: torch, segmentation-models-pytorch, onnx, onnxruntime.\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport sys\n\nimport numpy as np\n\nAQUI = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, os.path.join(AQUI, \'..\', \'backend\'))\nsys.path.insert(0, os.path.join(AQUI, \'..\'))          # repositorio v2: nucleo/ está en la raíz\n\nCANALES_6CH = [\'B02\', \'B03\', \'B04\', \'B08\', \'NDVI\', \'NDWI\']\nNOMBRES_14 = [\'B01\', \'B02\', \'B03\', \'B04\', \'B05\', \'B06\', \'B07\', \'B08\', \'B09\', \'B10\',\n              \'B11\', \'B12\', \'pendiente\', \'elevacion\']\nIDX_L4S = {n: i for i, n in enumerate(NOMBRES_14)}\nALIAS = {\'B13\': \'pendiente\', \'B14\': \'elevacion\', \'B1\': \'B01\', \'B2\': \'B02\', \'B3\': \'B03\',\n         \'B4\': \'B04\', \'B5\': \'B05\', \'B6\': \'B06\', \'B7\': \'B07\', \'B8\': \'B08\', \'B9\': \'B09\'}\n# Constantes de Landslide4Sense: pendiente = grados / 20.292, DEM = metros / 772.82\nESCALA_TERRENO_L4S = {\'pendiente\': 20.292, \'elevacion\': 772.82}\n\n\ndef construir(arquitectura, encoder, n_canales, n_clases=1):\n    import segmentation_models_pytorch as smp\n    clase = {\'deeplabv3plus\': smp.DeepLabV3Plus, \'unet\': smp.Unet}.get(arquitectura)\n    if clase is None:\n        raise SystemExit(f\'arquitectura "{arquitectura}" no reconocida\')\n    return clase(encoder_name=encoder, encoder_weights=None, in_channels=n_canales,\n                 classes=n_clases)\n\n\ndef con_tta(modelo, n):\n    """Envuelve el modelo: logits promediados sobre la imagen y sus espejos."""\n    import torch\n\n    class ConTTA(torch.nn.Module):\n        def __init__(self, m):\n            super().__init__()\n            self.m = m\n\n        def forward(self, x):\n            z = self.m(x)\n            if n >= 2:\n                z = z + torch.flip(self.m(torch.flip(x, [3])), [3])\n            if n >= 4:\n                z = z + torch.flip(self.m(torch.flip(x, [2])), [2])\n                z = z + torch.flip(self.m(torch.flip(x, [2, 3])), [2, 3])\n            return z / n\n\n    return ConTTA(modelo).eval()\n\n\ndef estado_del_checkpoint(ck):\n    """El state_dict, venga en el formato de la Fase 4 o suelto."""\n    if isinstance(ck, dict):\n        for clave in (\'modelo\', \'model_state_dict\', \'state_dict\', \'model\'):\n            if clave in ck and isinstance(ck[clave], dict):\n                return ck[clave]\n        if ck and all(hasattr(v, \'shape\') for v in ck.values()):\n            return ck\n    raise SystemExit(\'no encuentro los pesos en el checkpoint (claves: \'\n                     f\'{list(ck)[:8] if isinstance(ck, dict) else type(ck).__name__})\')\n\n\ndef constantes(stats, canales, metodo):\n    """Constantes de normalización por canal desde los formatos de --stats."""\n    out = {}\n    for i, c in enumerate(canales):\n        if c not in IDX_L4S:\n            continue                        # índices como NDVI: no se normalizan\n        if metodo == \'percentil\':\n            k = stats.get(str(IDX_L4S[c])) or stats.get(c)\n            if not k or \'lo\' not in k:\n                raise SystemExit(f\'--stats no trae lo y hi de {c}\')\n            out[c] = {\'lo\': float(k[\'lo\']), \'hi\': float(k[\'hi\'])}\n        else:\n            if \'mean\' in stats and \'std\' in stats:\n                out[c] = {\'media\': float(stats[\'mean\'][i]), \'desv\': float(stats[\'std\'][i])}\n            else:\n                k = stats.get(c) or stats.get(str(IDX_L4S[c]))\n                if not k or \'media\' not in k:\n                    raise SystemExit(f\'--stats no trae media y desv de {c}\')\n                out[c] = {\'media\': float(k[\'media\']), \'desv\': float(k[\'desv\'])}\n    return out\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--checkpoint\', required=True)\n    ap.add_argument(\'--stats\')\n    ap.add_argument(\'--resultados\')\n    ap.add_argument(\'--salida\', required=True)\n    ap.add_argument(\'--nombre\', required=True)\n    ap.add_argument(\'--etiqueta\')\n    ap.add_argument(\'--autor\')\n    ap.add_argument(\'--opset\', type=int, default=17)\n    ap.add_argument(\'--arquitectura\', choices=[\'unet\', \'deeplabv3plus\'])\n    ap.add_argument(\'--encoder\')\n    ap.add_argument(\'--lado\', type=int)\n    ap.add_argument(\'--canales\', help=\'p. ej. B01,B02,B03,B04,B13,B14\')\n    ap.add_argument(\'--clases\', type=int, choices=[1, 2],\n                    help=\'1: salida sigmoide; 2: softmax (fondo, deslizamiento)\')\n    ap.add_argument(\'--normalizacion\', choices=[\'percentil\', \'media_desv\', \'ninguna\'])\n    ap.add_argument(\'--umbral\', type=float)\n    ap.add_argument(\'--tta\', type=int, choices=[1, 4], default=1,\n                    help=\'4: el ONNX promedia la imagen y sus tres espejos (4 veces más lento)\')\n    a = ap.parse_args()\n\n    import torch\n    from nucleo import modelos as backend_modelos\n\n    ck = torch.load(a.checkpoint, map_location=\'cpu\', weights_only=False)\n    cfg = ck.get(\'config\', {}) if isinstance(ck, dict) else {}\n    arq = a.arquitectura or (ck.get(\'arquitectura\') if isinstance(ck, dict) else None)\n    if not arq:\n        raise SystemExit(\'el checkpoint no dice su arquitectura: usa --arquitectura\')\n    encoder = a.encoder or cfg.get(\'encoder\', \'resnet34\')\n    lado = int(a.lado or cfg.get(\'tam_parche\', 128))\n    if a.canales:\n        canales = [ALIAS.get(c.strip(), c.strip()) for c in a.canales.split(\',\') if c.strip()]\n    elif cfg.get(\'canales\'):            # checkpoints de entrenar.py\n        canales = [ALIAS.get(c, c) for c in cfg[\'canales\']]\n    elif cfg.get(\'config_canales\'):\n        canales = CANALES_6CH if cfg[\'config_canales\'] == \'6ch\' else NOMBRES_14\n    else:\n        raise SystemExit(\'el checkpoint no dice sus canales: usa --canales\')\n    desconocidos = [c for c in canales if c not in IDX_L4S and c not in (\'NDVI\', \'NDWI\')]\n    if desconocidos:\n        raise SystemExit(f\'canales desconocidos: {desconocidos}\')\n    n = len(canales)\n    estado = estado_del_checkpoint(ck)\n    if any(k.startswith(\'module.\') for k in estado):\n        estado = {k[len(\'module.\'):]: v for k, v in estado.items()}\n    cabeza = [v for k, v in estado.items() if \'segmentation_head\' in k and v.ndim == 4]\n    # versión 4: el modelo calcula dentro dos índices de verdor (entrenar.py --indices-verdor)\n    indices = bool((cfg.get(\'v4\') or {}).get(\'indices_verdor\')) or any(k.startswith(\'base.\') for k in estado)\n    n_clases = a.clases or (int(cabeza[0].shape[0]) if cabeza else 1)\n    print(f\'checkpoint : {os.path.basename(a.checkpoint)}\')\n    print(f\'modelo     : {arq}, codificador {encoder}, {n} canales {canales}, {lado}x{lado}, \'\n          f\'{n_clases} clase(s)\')\n    if isinstance(ck, dict):\n        print(f\'época {ck.get("epoca")}, IoU de validación {ck.get("iou_val")}, firma {ck.get("firma")}\')\n\n    modelo = construir(arq, encoder, n + (2 if indices else 0), n_clases)\n    if indices:\n        sys.path.insert(0, AQUI)\n        from entrenar import modelo_con_indices\n        modelo = modelo_con_indices(modelo, canales)\n        print(\'el modelo calcula dentro los índices de verdor GRVI y ExG (entra con los mismos canales)\')\n    antes = {k: v.clone() for k, v in modelo.state_dict().items()}\n    modelo.load_state_dict(estado, strict=True)\n    # solo los pesos que se inicializan al azar delatan si la carga ocurrió; los de\n    # BatchNorm arrancan en 1 y 0 en cualquier modelo nuevo\n    aleatorios = [k for k, v in antes.items() if v.dtype.is_floating_point and v.ndim >= 2]\n    cambiadas = sum(1 for k in aleatorios if not torch.equal(modelo.state_dict()[k], antes[k]))\n    print(f\'carga estricta: {cambiadas} de {len(aleatorios)} pesos aleatorios reemplazados\')\n    if cambiadas < len(aleatorios):\n        raise SystemExit(\'el checkpoint no reemplazó todos los pesos: no se exporta\')\n    modelo.eval()\n    if a.tta > 1:\n        modelo = con_tta(modelo, a.tta)\n        print(f\'TTA: el ONNX promedia {a.tta} versiones de cada parche\')\n\n    os.makedirs(a.salida, exist_ok=True)\n    ruta_onnx = os.path.join(a.salida, f\'{a.nombre}.onnx\')\n    x = torch.randn(2, n, lado, lado)\n    torch.onnx.export(modelo, x, ruta_onnx, input_names=[\'input\'], output_names=[\'output\'],\n                      dynamic_axes={\'input\': {0: \'N\'}, \'output\': {0: \'N\'}},\n                      opset_version=a.opset, do_constant_folding=True, dynamo=False)\n    print(f\'ONNX escrito: {ruta_onnx} ({os.path.getsize(ruta_onnx)/1e6:.1f} MB)\')\n\n    import onnxruntime as ort\n    ses = ort.InferenceSession(ruta_onnx, providers=[\'CPUExecutionProvider\'])\n    xs = torch.rand(4, n, lado, lado)\n    with torch.no_grad():\n        y_t = modelo(xs).numpy()\n    y_o = ses.run(None, {\'input\': xs.numpy()})[0]\n    dif = float(np.abs(y_t - y_o).max())\n    print(f\'paridad PyTorch / ONNX: diferencia máxima de logit {dif:.2e}\')\n    if dif > 1e-3:\n        raise SystemExit(\'el ONNX no reproduce a PyTorch\')\n\n    pesos = {k: v for k, v in backend_modelos.revisar_pesos(ruta_onnx).items()\n             if not k.startswith(\'_\')}\n    print(f\'prueba de pesos: {pesos}\')\n    if pesos[\'sin_entrenar\']:\n        raise SystemExit(\'el ONNX tiene pesos de inicialización: o el checkpoint no se cargó, \'\n                         \'o el modelo se entrenó tan pocos pasos que sus capas finales siguen \'\n                         \'como al inicio (pasa con 2 o 3 épocas de prueba, no con un \'\n                         \'entrenamiento completo)\')\n\n    metodo = a.normalizacion or cfg.get(\'normalizacion\') or \\\n        (\'percentil\' if cfg.get(\'config_canales\') else None)\n    if metodo is None:\n        raise SystemExit(\'di cómo se normalizó al entrenar: --normalizacion\')\n    if metodo != \'ninguna\' and not a.stats:\n        raise SystemExit(f\'--normalizacion {metodo} necesita --stats\')\n    stats = json.load(open(a.stats, encoding=\'utf-8\')) if a.stats else {}\n    por_canal = constantes(stats, canales, metodo) if metodo != \'ninguna\' else {}\n    res = json.load(open(a.resultados, encoding=\'utf-8\')) if a.resultados else {}\n    if a.tta == 4:\n        if \'tta4\' not in res:\n            raise SystemExit(\'--tta 4 necesita --resultados de entrenar.py versión 3 (trae el umbral con TTA)\')\n        res = dict(res, umbral_operacion=res[\'tta4\'][\'umbral_operacion\'],\n                   validacion=res[\'tta4\'][\'validacion\'], prueba=res[\'tta4\'][\'prueba\'])\n    terreno = [c for c in canales if c in ESCALA_TERRENO_L4S]\n    receta = {\n        \'nombre\': a.nombre,\n        \'etiqueta\': a.etiqueta or {\'deeplabv3plus\': \'DeepLabV3+\', \'unet\': \'U-Net\'}[arq]\n                    + f\' · {encoder}\',\n        \'autor\': a.autor,\n        \'onnx\': os.path.basename(ruta_onnx),\n        \'canales\': canales, \'lado_px\': lado,\n        \'normalizacion\': {\n            \'metodo\': metodo, \'percentiles\': cfg.get(\'percentiles\', [2, 98]),\n            \'canales\': list(por_canal),\n            \'por_canal\': por_canal,\n            \'nota\': \'constantes en unidades de Landslide4Sense; los índices NDVI y NDWI \'\n                    \'se calculan sobre esas unidades y no se normalizan\'},\n        \'salida\': \'sigmoide\' if n_clases == 1 else \'softmax\',\n        \'umbral\': float(a.umbral or res.get(\'umbral_operacion\', cfg.get(\'umbral_entren\', 0.5))),\n        **({\'tta\': a.tta} if a.tta > 1 else {}),\n        **({\'indices_internos\': \'GRVI y ExG de B03, B04 y B02, calculados dentro del ONNX\'} if indices else {}),\n        \'entrenamiento\': {\n            \'firma\': ck.get(\'firma\') if isinstance(ck, dict) else None,\n            \'arquitectura\': arq, \'codificador\': encoder,\n            \'dataset\': cfg.get(\'dataset\', \'Landslide4Sense\'),\n            \'epoca\': ck.get(\'epoca\') if isinstance(ck, dict) else None,\n            **({\'promedio_de_epocas\': ck[\'promedio_de_epocas\']}\n               if isinstance(ck, dict) and ck.get(\'promedio_de_epocas\') else {}),\n            \'iou_validacion\': res.get(\'validacion\', {}).get(\'iou\'),\n            \'prueba\': {k: res.get(\'prueba\', {}).get(k)\n                       for k in (\'iou\', \'f1\', \'precision\', \'recall\')},\n            \'particion\': cfg.get(\'particion\') or\n                         (\'grupos espaciales de TrainData (Fase 4)\' if cfg else None)},\n        **({\'rango_entrenamiento\': {c: {\'min\': stats[c][\'min\'], \'max\': stats[c][\'max\']}\n                                    for c in canales if isinstance(stats.get(c), dict)\n                                    and \'min\' in stats[c]}}\n           if any(isinstance(stats.get(c), dict) and \'min\' in stats[c] for c in canales) else {}),\n        **({\'escala_terreno\': {c: ESCALA_TERRENO_L4S[c] for c in terreno},\n            \'terreno_entrenamiento\': \'ALOS PALSAR (ASF) en Landslide4Sense: B13 pendiente, \'\n                                     \'B14 DEM\'} if terreno else {}),\n        \'exportacion\': {\'opset\': a.opset, \'paridad_max_logit\': dif, \'pesos\': pesos,\n                        \'sha256_onnx\': hashlib.sha256(open(ruta_onnx, \'rb\').read()).hexdigest()},\n    }\n    ruta_json = os.path.join(a.salida, f\'{a.nombre}.json\')\n    json.dump(receta, open(ruta_json, \'w\', encoding=\'utf-8\'), indent=1, ensure_ascii=False)\n    print(f\'receta escrita: {ruta_json}\')\n\n\nif __name__ == \'__main__\':\n    main()\n'
ARCHIVOS['herramientas/verificar_onnx.py'] = '"""\nVerifica un ONNX con su receta sobre parches de Landslide4Sense con máscara.\n\nUsa la misma función que el backend (`recetas.desde_l4s` -> `recetas.armar`),\nasí que lo que se mide aquí es lo que corre en producción, no una copia.\n\n  python3 verificar_onnx.py --modelos ../backend/modelos --l4s /ruta/l4s \\\n      [--splits validation test] [--lista split_grupos.csv --grupo test] \\\n      [--escribir]\n\n--l4s apunta a la estructura del dataset de Hugging Face\n(ibm-nasa-geospatial/Landslide4sense): images/<split>/image_N.h5 y\nannotations/<split>/mask_N.h5.\n--escribir guarda el resultado en la receta (campo verificacion_l4s), que el\nbackend publica en /modelos y en el manifiesto de cada ZIP.\n"""\nimport argparse\nimport datetime as dt\nimport glob\nimport json\nimport os\nimport sys\n\nimport numpy as np\n\nAQUI = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, os.path.join(AQUI, \'..\', \'backend\'))\nsys.path.insert(0, os.path.join(AQUI, \'..\'))          # repositorio v2: nucleo/ está en la raíz\n\n\ndef pares(raiz, split, nombres=None):\n    imgs = glob.glob(os.path.join(raiz, \'images\', split, \'image_*.h5\'))\n    imgs.sort(key=lambda p: int(p.rsplit(\'_\', 1)[1][:-3]))\n    for p in imgs:\n        if nombres is not None and os.path.basename(p) not in nombres:\n            continue\n        q = p.replace(os.sep + \'images\' + os.sep, os.sep + \'annotations\' + os.sep) \\\n             .replace(\'image_\', \'mask_\')\n        if os.path.exists(q):\n            yield p, q\n\n\ndef evaluar(ses, receta, lista, lote=16):\n    import h5py\n    from nucleo import recetas\n    tp = fp = fn = tn = 0\n    ious = []\n    nombre = ses.get_inputs()[0].name\n    for i in range(0, len(lista), lote):\n        X, Y, V = [], [], []\n        for p, q in lista[i:i + lote]:\n            with h5py.File(p, \'r\') as f:\n                t, v = recetas.desde_l4s(f[\'img\'][:], receta)\n            with h5py.File(q, \'r\') as f:\n                y = np.asarray(f[\'mask\'][:]) > 0\n            X.append(t); Y.append(y & v); V.append(v)\n        prob = recetas.probabilidad(ses.run(None, {nombre: np.stack(X)})[0], receta[\'salida\'])\n        for pk, y, v in zip(prob, Y, V):\n            m = (pk >= receta[\'umbral\']) & v\n            a, b, c = int((m & y).sum()), int((m & ~y).sum()), int((~m & y & v).sum())\n            tp += a; fp += b; fn += c; tn += int((~m & ~y & v).sum())\n            if y.any():\n                ious.append(a / max(a + b + c, 1))\n    iou = tp / max(tp + fp + fn, 1)\n    pr = tp / max(tp + fp, 1); rc = tp / max(tp + fn, 1)\n    return {\'parches\': len(lista), \'iou\': round(iou, 4),\n            \'f1\': round(2 * pr * rc / max(pr + rc, 1e-9), 4),\n            \'precision\': round(pr, 4), \'recall\': round(rc, 4),\n            \'iou_medio_parches_con_deslizamiento\': round(float(np.mean(ious)), 4) if ious else None,\n            \'tp\': tp, \'fp\': fp, \'fn\': fn, \'tn\': tn}\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--modelos\', required=True)\n    ap.add_argument(\'--l4s\', required=True)\n    ap.add_argument(\'--splits\', nargs=\'+\', default=[\'validation\', \'test\'])\n    ap.add_argument(\'--lista\', help=\'CSV de la Fase 4 con columnas de archivo y split\')\n    ap.add_argument(\'--grupo\', default=\'test\')\n    ap.add_argument(\'--escribir\', action=\'store_true\')\n    a = ap.parse_args()\n\n    import onnxruntime as ort\n    from nucleo import recetas\n    from nucleo.modelos import revisar_pesos\n\n    for ruta_r in sorted(glob.glob(os.path.join(a.modelos, \'*.json\'))):\n        if json.load(open(ruta_r, encoding=\'utf-8\')).get(\'tipo\') == \'ensamble\':\n            print(f\'\\n===== {os.path.basename(ruta_r)}: ensamble, sin ONNX propio; se salta\')\n            continue\n        r = recetas.leer(ruta_r)\n        onnx = os.path.join(a.modelos, r[\'onnx\'])\n        print(f\'\\n===== {r["nombre"]} ({os.path.basename(onnx)})\')\n        pesos = {k: v for k, v in revisar_pesos(onnx).items() if not k.startswith(\'_\')}\n        print(f\'  pesos: {pesos}\')\n        if pesos[\'sin_entrenar\']:\n            print(\'  SIN ENTRENAR: no tiene sentido medirlo\'); continue\n        so = ort.SessionOptions(); so.intra_op_num_threads = os.cpu_count() or 2\n        ses = ort.InferenceSession(onnx, so, providers=[\'CPUExecutionProvider\'])\n        resultado = {\'fecha\': dt.date.today().isoformat(), \'umbral\': r[\'umbral\'],\n                     \'dataset\': \'ibm-nasa-geospatial/Landslide4sense (Hugging Face)\'}\n        if a.lista:\n            import pandas as pd\n            t = pd.read_csv(a.lista)\n            col_split = \'split\' if \'split\' in t.columns else t.columns[-1]\n            col_img = next(c for c in t.columns if \'img\' in c.lower() or \'imag\' in c.lower())\n            sel = t[t[col_split] == a.grupo]\n            nombres = {os.path.basename(str(x)) for x in sel[col_img]}\n            lista = list(pares(a.l4s, \'train\', nombres))\n            m = evaluar(ses, r, lista)\n            print(f\'  {a.grupo} de la Fase 4 ({len(lista)} de {len(nombres)} parches): {m}\')\n            resultado[f\'fase4_{a.grupo}\'] = m\n        for s in a.splits:\n            lista = list(pares(a.l4s, s))\n            if not lista:\n                continue\n            m = evaluar(ses, r, lista)\n            print(f\'  {s}: {m}\')\n            resultado[s] = m\n        ref = (r.get(\'entrenamiento\') or {}).get(\'prueba\', {})\n        if ref.get(\'iou\') is not None:\n            print(f\'  referencia del entrenamiento, prueba: IoU {ref["iou"]:.4f}, F1 {ref["f1"]:.4f}\')\n            medidos = [resultado[s][\'iou\'] for s in a.splits if s in resultado]\n            if medidos and max(medidos) < 0.6 * ref[\'iou\']:\n                print(\'  ATENCIÓN: el IoU medido queda muy por debajo del de entrenamiento. \'\n                      \'Revisa que el checkpoint, las estadísticas y la receta sean de la misma \'\n                      \'firma; con un ONNX bien exportado la diferencia es pequeña.\')\n                resultado[\'alerta\'] = \'IoU muy por debajo del de entrenamiento\'\n        if a.escribir:\n            crudo = json.load(open(ruta_r, encoding=\'utf-8\'))\n            crudo[\'verificacion_l4s\'] = resultado\n            json.dump(crudo, open(ruta_r, \'w\', encoding=\'utf-8\'), indent=1, ensure_ascii=False)\n            print(f\'  escrito en {os.path.basename(ruta_r)}\')\n\n\nif __name__ == \'__main__\':\n    main()\n'

for ruta, contenido in ARCHIVOS.items():
    ruta_abs = os.path.join('/content/artefacto', ruta)
    os.makedirs(os.path.dirname(ruta_abs), exist_ok=True)
    open(ruta_abs, 'w', encoding='utf-8').write(contenido)
print('Código del proyecto escrito en /content/artefacto')


## Landslide4Sense

Los tres conjuntos oficiales (~9 GB). Sirven de ancla al entrenar y de prueba al final.

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('ibm-nasa-geospatial/Landslide4sense', repo_type='dataset',
                  local_dir='/content/l4s', max_workers=16)
for d in ('images/train', 'annotations/train', 'images/validation', 'annotations/validation',
          'images/test', 'annotations/test'):
    print(d, len(os.listdir(f'/content/l4s/{d}')))

## Los parches de La Convención

In [ ]:
shutil.rmtree('/content/local', ignore_errors=True)
zipfile.ZipFile(ZIP_LOCAL).extractall('/content/local')
import csv, collections
ruta_n1 = '/content/local/mascaras_nivel1/mascaras_nivel1.csv'
if os.path.exists(ruta_n1):
    resumen = list(csv.DictReader(open(ruta_n1, encoding='utf-8')))
    c = collections.Counter(r['conjunto'] for r in resumen)
    print('Nivel 1 (marcado humano) por conjunto:', dict(c))
    for conj in ('entrenamiento', 'validacion'):
        f = [r for r in resumen if r['conjunto'] == conj]
        print(f"  {conj}: {len(f)} teselas, {sum(int(r['cicatrices']) for r in f)} cicatrices, "
              f"{sum(float(r['ha_deslizamiento']) for r in f):.1f} ha, "
              f"{sum(float(r['ha_deslizamiento']) == 0 for r in f)} negativos confirmados, "
              f"{len({r['zona'] for r in f})} zonas")
else:
    print('Nivel 1: 0 máscaras humanas exportadas aún (entrenando con Tiers de Nivel 2).')

ruta_tiers = '/content/local/mascaras_nivel2/catalogo_parches_tiers.csv'
if os.path.exists(ruta_tiers):
    tiers = list(csv.DictReader(open(ruta_tiers, encoding='utf-8')))
    ent_t = [r for r in tiers if r['conjunto'] == 'entrenamiento' and float(r.get('peso_perdida') or 0) > 0]
    print('Nivel 2 (Tiers de entrenamiento):', dict(collections.Counter(r['tier'] for r in ent_t)))


In [ ]:
def correr(*args):
    # Corre un script mostrando su salida en vivo; si falla, el cuaderno se detiene.
    p = subprocess.Popen([sys.executable, *args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    cola = []
    for linea in p.stdout:
        cola = (cola + [linea])[-60:]
        if 'Warning' not in linea and 'warn(' not in linea:
            print(linea, end='')
    if p.wait():
        print(''.join(cola[-30:]))
        raise RuntimeError('falló; revisa el mensaje de arriba')

## Ajuste fino

Cada época muestra el IoU en la validación de Landslide4Sense y en la validación local. La época 0
es el modelo base sin tocar.

In [ ]:
antes = set(glob.glob(f'{SALIDA}/resultados_*.json'))
args = ['/content/artefacto/herramientas/ajuste_fino.py', '--l4s', '/content/l4s',
        '--desde', CK_BASE, '--stats', STATS_BASE,
        '--parches', '/content/local/parches_128',
        '--mascaras', '/content/local/mascaras_nivel1/parches_128',
        '--escala', '/content/artefacto/escala_laconvencion.json',
        '--salida', SALIDA, '--epocas', str(EPOCAS), '--lr', str(TASA),
        '--frac-local', str(FRACCION_LOCAL), '--tolerancia-l4s', str(TOLERANCIA_L4S)]
if CONGELAR_ENCODER:
    args.append('--congelar-encoder')
if USAR_DEBILES:
    args += ['--debiles', '/content/local/mascaras_nivel2/parches_128', '--peso-debil', str(PESO_DEBIL)]
if USAR_GEOMORFOLOGIA:
    args.append('--geomorfologia')
correr(*args)
nuevos = sorted(set(glob.glob(f'{SALIDA}/resultados_*.json')) - antes, key=os.path.getmtime)
RES = nuevos[-1] if nuevos else sorted(glob.glob(f'{SALIDA}/resultados_*.json'), key=os.path.getmtime)[-1]
print('resultados:', RES)


## Antes y después

In [ ]:
r = json.load(open(RES, encoding='utf-8'))
fila = lambda n, m: print(f"{n:38s} IoU {m['iou']:.4f}  F1 {m['f1']:.4f}  precisión {m['precision']*100:5.1f} %  recall {m['recall']*100:5.1f} %")
print('PRUEBA OFICIAL DE LANDSLIDE4SENSE (800 parches)')
if 'comparador_sin_aprendizaje' in r:
    fila('  regla sin aprendizaje (GRVI+pend)', r['comparador_sin_aprendizaje']['l4s_prueba'])
fila('  U-Net v4 (base)', r['base']['l4s_prueba']); fila('  ajustada', r['prueba'])
if 'local_validacion' in r:
    b, j = r['base']['local_validacion'], r['local_validacion']
    print(f"\nVALIDACIÓN LOCAL ({j['teselas']} teselas, {j['cicatrices']} cicatrices, {j['teselas_negativas']} teselas negativas)")
    if 'comparador_sin_aprendizaje' in r and 'local_validacion' in r['comparador_sin_aprendizaje']:
        rg = r['comparador_sin_aprendizaje']['local_validacion']
        fila('  regla sin aprendizaje (GRVI+pend)', rg)
    fila(f"  U-Net v4 (base), umbral {b['umbral']}", b); fila(f"  ajustada, umbral {j['umbral']}", j)
    print(f"  cicatrices encontradas: base {b['cicatrices_tocadas']} de {b['cicatrices']}, ajustada {j['cicatrices_tocadas']} de {j['cicatrices']}")
    print(f"  falsas alarmas en teselas negativas: base {b['falsas_alarmas_ha_por_km2_en_negativos']} ha/km², "
          f"ajustada {j['falsas_alarmas_ha_por_km2_en_negativos']} ha/km²")
    if r.get('mejoro') and r.get('local_mejora_iou'):
        z = r['local_mejora_iou']
        print(f"  mejora de IoU, IC 95 % remuestreando {z['zonas']} zonas: [{z['ic95'][0]:+.4f}, {z['ic95'][1]:+.4f}]")
        if z['ic95'][0] <= 0:
            print('  -> el intervalo incluye el cero: con estos datos NO se puede afirmar que mejoró')
    for s in r.get('local_advertencias', []):
        print('  ojo:', s)
print()
if not r['mejoro']:
    print('RESULTADO: se queda el modelo base. No hay nada que exportar; hacen falta más teselas marcadas.')
else:
    perdida = r['base']['l4s_prueba']['iou'] - r['prueba']['iou']
    print(f"RESULTADO: modelo ajustado en la época {r['mejor_epoca']}, umbral {r['umbral_operacion']} "
          f"(elegido en {r['umbral_elegido_en']}). En Landslide4Sense "
          + (f"pierde {perdida:.4f} de IoU." if perdida > 0 else f"gana {-perdida:.4f} de IoU."))

## Exportar a ONNX (solo si mejoró)

Deja `unet_ajustada.onnx` y su receta en Drive. **No reemplaza nada en Render**: subirlo al
repositorio es una decisión aparte, que conviene tomar cuando la validación local tenga
cicatrices suficientes.

In [ ]:
if r['mejoro']:
    firma = r['firma']
    os.makedirs('/content/onnx', exist_ok=True)
    correr('/content/artefacto/herramientas/exportar_onnx.py',
           '--checkpoint', f'{SALIDA}/mejor_unet_b1b4b13b14_{firma}.pt',
           '--stats', f'{SALIDA}/stats_norm_{firma}.json', '--resultados', RES,
           '--salida', '/content/onnx', '--nombre', 'unet_ajustada',
           '--etiqueta', 'U-Net · resnet34 · ajustada a La Convención', '--autor', AUTOR)
    correr('/content/artefacto/herramientas/verificar_onnx.py', '--modelos', '/content/onnx',
           '--l4s', '/content/l4s', '--splits', 'validation', 'test', '--escribir')
    os.makedirs(f'{SALIDA}/onnx', exist_ok=True)
    for f in os.listdir('/content/onnx'):
        shutil.copy(os.path.join('/content/onnx', f), f'{SALIDA}/onnx')
    print('guardado en', f'{SALIDA}/onnx', sorted(os.listdir(f'{SALIDA}/onnx')))
else:
    print('no se exporta: el modelo base sigue siendo el mejor')